# BigAlpha 2026 current Elastic Net v04

当前 I 路线冻结池；正系数 Elastic Net；平台安全查询边界，J baseline 统一为 factorlib all36。

In [ ]:
"""Current I-route submission v04: refreshed Elastic Net pool."""

# Auto-generated from current remote training artifacts. Do not edit by hand.

def _install_bigalpha_candidate_modules():
    import sys
    import types
    sources = {'bigalpha2026.candidates.fr._common': '"""Shared PIT helpers for financial-report candidates."""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Iterable\n\nimport numpy as np\nimport pandas as pd\n\nPOOL_COLUMNS = ("date", "instrument")\nOUTPUT_COLUMNS = ("date", "instrument", "factor")\n\n\ndef require_columns(\n    frame: pd.DataFrame,\n    columns: Iterable[str],\n    name: str,\n) -> None:\n    missing = sorted(set(columns).difference(frame.columns))\n    if missing:\n        raise ValueError(f"{name} is missing required columns: {missing}")\n\n\ndef group_asof(\n    left: pd.DataFrame,\n    right: pd.DataFrame,\n    *,\n    left_on: str,\n    right_on: str,\n    right_columns: list[str],\n) -> pd.DataFrame:\n    left_frame = left.copy()\n    left_frame[left_on] = pd.to_datetime(\n        left_frame[left_on], errors="coerce"\n    ).astype("datetime64[ns]")\n    right_frame = right.copy()\n    right_frame[right_on] = pd.to_datetime(\n        right_frame[right_on], errors="coerce"\n    ).astype("datetime64[ns]")\n    left_frame["_row_order"] = np.arange(len(left_frame))\n    right_groups = {\n        instrument: block.sort_values(right_on)\n        for instrument, block in right_frame.groupby("instrument", sort=False)\n    }\n    pieces: list[pd.DataFrame] = []\n    for instrument, left_block in left_frame.groupby("instrument", sort=False):\n        ordered = left_block.sort_values(left_on)\n        right_block = right_groups.get(instrument)\n        if right_block is None or right_block.empty:\n            for column in [right_on, *right_columns]:\n                ordered[column] = pd.NaT if column == right_on else np.nan\n            pieces.append(ordered)\n            continue\n        pieces.append(\n            pd.merge_asof(\n                ordered,\n                right_block[[right_on, *right_columns]].sort_values(right_on),\n                left_on=left_on,\n                right_on=right_on,\n                direction="backward",\n                allow_exact_matches=True,\n            )\n        )\n    if not pieces:\n        return left_frame.drop(columns="_row_order")\n    return (\n        pd.concat(pieces, ignore_index=True)\n        .sort_values("_row_order")\n        .drop(columns="_row_order")\n        .reset_index(drop=True)\n    )\n\n\ndef prepare_event_panel(\n    financial: pd.DataFrame,\n    *,\n    category: str,\n    value_column: str,\n) -> pd.DataFrame:\n    required = (\n        "disclosure_date",\n        "effective_date",\n        "instrument",\n        "report_date",\n        "category",\n        "shift",\n        value_column,\n    )\n    require_columns(financial, required, "financial")\n    frame = financial.loc[:, required].copy()\n    for column in ("disclosure_date", "effective_date", "report_date"):\n        frame[column] = pd.to_datetime(frame[column], errors="coerce").dt.normalize()\n    frame["instrument"] = frame["instrument"].astype(str)\n    frame["category"] = frame["category"].astype(str).str.lower()\n    frame["shift"] = pd.to_numeric(frame["shift"], errors="coerce")\n    frame[value_column] = pd.to_numeric(frame[value_column], errors="coerce")\n    return (\n        frame.loc[frame["category"].eq(category.lower()) & frame["shift"].eq(0)]\n        .dropna(\n            subset=[\n                "disclosure_date",\n                "effective_date",\n                "instrument",\n                "report_date",\n            ]\n        )\n        .sort_values(["instrument", "disclosure_date", "report_date"])\n        .drop_duplicates(["disclosure_date", "instrument", "report_date"], keep="last")\n        .reset_index(drop=True)\n    )\n\n\ndef year_over_year_events(\n    financial: pd.DataFrame,\n    *,\n    category: str,\n    value_column: str,\n    output_column: str,\n    sign: float,\n) -> pd.DataFrame:\n    """Compute a PIT year-over-year change using only disclosures known then."""\n\n    frame = prepare_event_panel(\n        financial,\n        category=category,\n        value_column=value_column,\n    )\n    pieces: list[pd.DataFrame] = []\n    for _, block in frame.groupby("instrument", sort=False):\n        history: dict[pd.Timestamp, float] = {}\n        values: list[float] = []\n        for row in block.itertuples(index=False):\n            report_date = pd.Timestamp(row.report_date)\n            current = getattr(row, value_column)\n            previous = history.get(report_date - pd.DateOffset(years=1))\n            if (\n                previous is None\n                or not np.isfinite(previous)\n                or abs(previous) <= 1e-12\n                or not np.isfinite(current)\n            ):\n                values.append(np.nan)\n            else:\n                values.append(sign * (current - previous) / abs(previous))\n            if np.isfinite(current):\n                history[report_date] = float(current)\n        enriched = block.copy()\n        enriched[output_column] = values\n        pieces.append(enriched)\n    if not pieces:\n        frame[output_column] = np.nan\n        return frame\n    events = pd.concat(pieces, ignore_index=True)\n    events[output_column] = pd.to_numeric(\n        events[output_column],\n        errors="coerce",\n    ).replace([np.inf, -np.inf], np.nan)\n    return (\n        events.sort_values(\n            ["instrument", "effective_date", "report_date", "disclosure_date"]\n        )\n        .drop_duplicates(["instrument", "effective_date"], keep="last")\n        .reset_index(drop=True)\n    )\n\n\ndef prepare_pool(pool: pd.DataFrame) -> pd.DataFrame:\n    require_columns(pool, POOL_COLUMNS, "pool")\n    panel = pool.loc[:, POOL_COLUMNS].copy()\n    panel["date"] = pd.to_datetime(panel["date"], errors="coerce").dt.normalize().astype("datetime64[ns]")\n    panel["instrument"] = panel["instrument"].astype(str)\n    panel = panel.dropna(subset=list(POOL_COLUMNS))\n    if panel.duplicated(list(POOL_COLUMNS)).any():\n        raise ValueError("pool contains duplicate date-instrument keys")\n    return panel\n\n\ndef rank_state(state: pd.DataFrame, *, candidate_id: str) -> pd.DataFrame:\n    require_columns(state, (*POOL_COLUMNS, "factor_raw"), "state")\n    output = state.copy()\n    output["factor_raw"] = pd.to_numeric(\n        output["factor_raw"],\n        errors="coerce",\n    ).replace([np.inf, -np.inf], np.nan)\n    median = output.groupby("date", sort=False)["factor_raw"].transform("median")\n    output["factor_raw"] = output["factor_raw"].fillna(median).fillna(0.0)\n    output["factor"] = (\n        output.groupby("date", sort=False)["factor_raw"]\n        .rank(pct=True, method="average")\n        .sub(0.5)\n        .mul(2.0)\n    )\n    if not np.isfinite(output["factor"]).all():\n        raise ValueError(f"{candidate_id} produced non-finite factor values")\n    return (\n        output.loc[:, OUTPUT_COLUMNS]\n        .sort_values(["date", "instrument"])\n        .reset_index(drop=True)\n    )\n\n\ndef build_event_factor(\n    events: pd.DataFrame,\n    pool: pd.DataFrame,\n    *,\n    value_column: str,\n    candidate_id: str,\n) -> pd.DataFrame:\n    panel = prepare_pool(pool)\n    state = group_asof(\n        panel,\n        events,\n        left_on="date",\n        right_on="effective_date",\n        right_columns=[value_column],\n    )\n    state["factor_raw"] = state[value_column]\n    return rank_state(state, candidate_id=candidate_id)\n', 'bigalpha2026.candidates.pv._common': '"""Shared helpers for daily price-volume candidates."""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Iterable\n\nimport numpy as np\nimport pandas as pd\n\nPOOL_COLUMNS = ("date", "instrument")\nOUTPUT_COLUMNS = ("date", "instrument", "factor")\n\n\ndef require_columns(\n    frame: pd.DataFrame,\n    columns: Iterable[str],\n    name: str,\n) -> None:\n    missing = sorted(set(columns).difference(frame.columns))\n    if missing:\n        raise ValueError(f"{name} is missing required columns: {missing}")\n\n\ndef prepare_daily(\n    daily_bars: pd.DataFrame,\n    required: Iterable[str],\n) -> pd.DataFrame:\n    required = tuple(required)\n    require_columns(daily_bars, required, "daily_bars")\n    frame = daily_bars.loc[:, required].copy()\n    frame["date"] = pd.to_datetime(frame["date"], errors="coerce").dt.normalize()\n    frame["instrument"] = frame["instrument"].astype(str)\n    for column in required:\n        if column not in POOL_COLUMNS:\n            frame[column] = pd.to_numeric(frame[column], errors="coerce")\n    frame = frame.dropna(subset=list(POOL_COLUMNS))\n    if frame.duplicated(list(POOL_COLUMNS)).any():\n        raise ValueError("daily_bars contains duplicate date-instrument keys")\n    return frame.sort_values(["instrument", "date"]).reset_index(drop=True)\n\n\ndef rolling_by_instrument(\n    frame: pd.DataFrame,\n    column: str,\n    *,\n    window: int,\n    min_periods: int,\n    method: str,\n) -> pd.Series:\n    if window < 2:\n        raise ValueError("window must be at least 2")\n    if min_periods < 2 or min_periods > window:\n        raise ValueError("min_periods must be between 2 and window")\n    rolling = frame.groupby("instrument", sort=False)[column].rolling(\n        window,\n        min_periods=min_periods,\n    )\n    if method == "max":\n        values = rolling.max()\n    elif method == "mean":\n        values = rolling.mean()\n    elif method == "skew":\n        values = rolling.skew()\n    else:\n        raise ValueError(f"unsupported rolling method: {method}")\n    return values.reset_index(level=0, drop=True).sort_index()\n\n\ndef build_ranked_factor(\n    features: pd.DataFrame,\n    pool: pd.DataFrame,\n    *,\n    candidate_id: str,\n    start_date: object | None = None,\n    end_date: object | None = None,\n) -> pd.DataFrame:\n    require_columns(pool, POOL_COLUMNS, "pool")\n    require_columns(features, (*POOL_COLUMNS, "factor_raw"), "features")\n    panel = pool.loc[:, POOL_COLUMNS].copy()\n    panel["date"] = pd.to_datetime(panel["date"], errors="coerce").dt.normalize().astype("datetime64[ns]")\n    panel["instrument"] = panel["instrument"].astype(str)\n    panel = panel.dropna(subset=list(POOL_COLUMNS))\n    if panel.duplicated(list(POOL_COLUMNS)).any():\n        raise ValueError("pool contains duplicate date-instrument keys")\n    if start_date is not None:\n        panel = panel.loc[panel["date"] >= pd.Timestamp(start_date).normalize()]\n    if end_date is not None:\n        panel = panel.loc[panel["date"] <= pd.Timestamp(end_date).normalize()]\n\n    values = features.loc[:, [*POOL_COLUMNS, "factor_raw"]].copy()\n    values["date"] = pd.to_datetime(values["date"], errors="coerce").dt.normalize().astype("datetime64[ns]")\n    values["instrument"] = values["instrument"].astype(str)\n    if values.duplicated(list(POOL_COLUMNS)).any():\n        raise ValueError("features contains duplicate date-instrument keys")\n    result = panel.merge(\n        values,\n        on=list(POOL_COLUMNS),\n        how="left",\n        validate="one_to_one",\n    )\n    result["factor_raw"] = pd.to_numeric(\n        result["factor_raw"],\n        errors="coerce",\n    ).replace([np.inf, -np.inf], np.nan)\n    daily_median = result.groupby("date", sort=False)["factor_raw"].transform("median")\n    result["factor_raw"] = result["factor_raw"].fillna(daily_median).fillna(0.0)\n    result["factor"] = (\n        result.groupby("date", sort=False)["factor_raw"]\n        .rank(pct=True, method="average")\n        .sub(0.5)\n        .mul(2.0)\n    )\n    if not np.isfinite(result["factor"]).all():\n        raise ValueError(f"{candidate_id} produced non-finite factor values")\n    return (\n        result.loc[:, OUTPUT_COLUMNS]\n        .sort_values(["date", "instrument"])\n        .reset_index(drop=True)\n    )\n', 'bigalpha2026.candidates.pv._monthly': '"""Shared monthly-state helpers for long-horizon PV candidates."""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom ._common import build_ranked_factor, prepare_daily, require_columns\n\n\ndef monthly_values(\n    daily_bars: pd.DataFrame,\n    *,\n    value_column: str,\n    aggregation: str,\n) -> pd.DataFrame:\n    frame = prepare_daily(\n        daily_bars,\n        ("date", "instrument", value_column),\n    )\n    frame["month"] = frame["date"].dt.to_period("M")\n    grouped = frame.groupby(["instrument", "month"], sort=False)\n    if aggregation == "sum":\n        values = grouped[value_column].sum(min_count=1)\n    elif aggregation == "mean":\n        values = grouped[value_column].mean()\n    else:\n        raise ValueError(f"unsupported monthly aggregation: {aggregation}")\n    dates = grouped["date"].max()\n    return (\n        pd.concat([dates.rename("effective_date"), values.rename("value")], axis=1)\n        .reset_index()\n        .sort_values(["instrument", "effective_date"])\n        .reset_index(drop=True)\n    )\n\n\ndef build_monthly_state_factor(\n    monthly: pd.DataFrame,\n    pool: pd.DataFrame,\n    *,\n    candidate_id: str,\n) -> pd.DataFrame:\n    require_columns(\n        monthly,\n        ("effective_date", "instrument", "factor_raw"),\n        "monthly",\n    )\n    require_columns(pool, ("date", "instrument"), "pool")\n    panel = pool[["date", "instrument"]].copy()\n    panel["date"] = pd.to_datetime(panel["date"], errors="coerce").dt.normalize().astype("datetime64[ns]")\n    panel["instrument"] = panel["instrument"].astype(str)\n    events = monthly[\n        ["effective_date", "instrument", "factor_raw"]\n    ].copy()\n    events["effective_date"] = pd.to_datetime(\n        events["effective_date"],\n        errors="coerce",\n    ).dt.normalize().astype("datetime64[ns]")\n    events["instrument"] = events["instrument"].astype(str)\n    event_groups = {\n        instrument: block.sort_values("effective_date")\n        for instrument, block in events.groupby("instrument", sort=False)\n    }\n    pieces: list[pd.DataFrame] = []\n    for instrument, block in panel.groupby("instrument", sort=False):\n        ordered = block.sort_values("date")\n        event_block = event_groups.get(instrument)\n        if event_block is None or event_block.empty:\n            ordered["factor_raw"] = np.nan\n        else:\n            ordered = pd.merge_asof(\n                ordered,\n                event_block[["effective_date", "factor_raw"]],\n                left_on="date",\n                right_on="effective_date",\n                direction="backward",\n            ).drop(columns="effective_date")\n        pieces.append(ordered)\n    daily = pd.concat(pieces, ignore_index=True)\n    return build_ranked_factor(\n        daily,\n        panel,\n        candidate_id=candidate_id,\n    )\n', 'bigalpha2026.candidates.fr.fr_001': '"""FR-001: newly disclosed cash-conversion improvement.\n\nFinancial observations become usable only on the first CN trading day strictly\nafter their disclosure date. Empirical evaluation belongs in AIStudio.\n"""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Iterable\n\nimport numpy as np\nimport pandas as pd\n\nFINANCIAL_COLUMNS = (\n    "date",\n    "instrument",\n    "category",\n    "shift",\n    "report_date",\n    "net_cffoa",\n    "net_profit",\n    "total_assets",\n)\nPOOL_COLUMNS = ("date", "instrument")\nOUTPUT_COLUMNS = ("date", "instrument", "factor")\n\n\ndef _require_columns(frame: pd.DataFrame, columns: Iterable[str], name: str) -> None:\n    missing = sorted(set(columns).difference(frame.columns))\n    if missing:\n        raise ValueError(f"{name} is missing required columns: {missing}")\n\n\ndef _group_asof(\n    left: pd.DataFrame,\n    right: pd.DataFrame,\n    *,\n    left_on: str,\n    right_on: str,\n    right_columns: list[str],\n) -> pd.DataFrame:\n    """Backward as-of merge within each instrument, preserving left order."""\n\n    left_frame = left.copy()\n    left_frame[left_on] = pd.to_datetime(\n        left_frame[left_on], errors="coerce"\n    ).astype("datetime64[ns]")\n    right_frame = right.copy()\n    right_frame[right_on] = pd.to_datetime(\n        right_frame[right_on], errors="coerce"\n    ).astype("datetime64[ns]")\n    left_frame["_row_order"] = np.arange(len(left_frame))\n    right_groups = {\n        instrument: block.sort_values(right_on)\n        for instrument, block in right_frame.groupby("instrument", sort=False)\n    }\n    pieces: list[pd.DataFrame] = []\n    for instrument, left_block in left_frame.groupby("instrument", sort=False):\n        ordered = left_block.sort_values(left_on)\n        right_block = right_groups.get(instrument)\n        if right_block is None or right_block.empty:\n            for column in [right_on, *right_columns]:\n                ordered[column] = pd.NaT if column == right_on else np.nan\n            pieces.append(ordered)\n            continue\n        merged = pd.merge_asof(\n            ordered,\n            right_block[[right_on, *right_columns]].sort_values(right_on),\n            left_on=left_on,\n            right_on=right_on,\n            direction="backward",\n            allow_exact_matches=True,\n        )\n        pieces.append(merged)\n    if not pieces:\n        return left_frame.drop(columns="_row_order")\n    return (\n        pd.concat(pieces, ignore_index=True)\n        .sort_values("_row_order")\n        .drop(columns="_row_order")\n        .reset_index(drop=True)\n    )\n\n\ndef compute_fr_001_events(\n    financial: pd.DataFrame,\n    trading_days: Iterable[object],\n) -> pd.DataFrame:\n    """Build PIT disclosure events with TTM flows and latest available LF assets."""\n\n    if "date" not in financial.columns and "disclosure_date" in financial.columns:\n        financial = financial.rename(columns={"disclosure_date": "date"})\n    _require_columns(financial, FINANCIAL_COLUMNS, "financial")\n    calendar = pd.DatetimeIndex(pd.to_datetime(list(trading_days), errors="coerce"))\n    calendar = calendar.dropna().normalize().unique().sort_values()\n    if len(calendar) == 0:\n        raise ValueError("trading_days must contain at least one valid date")\n\n    frame = financial.loc[:, FINANCIAL_COLUMNS].copy()\n    frame["date"] = pd.to_datetime(frame["date"], errors="coerce").dt.normalize()\n    frame["report_date"] = pd.to_datetime(\n        frame["report_date"],\n        errors="coerce",\n    ).dt.normalize()\n    frame["instrument"] = frame["instrument"].astype(str)\n    frame["category"] = frame["category"].astype(str).str.lower()\n    frame["shift"] = pd.to_numeric(frame["shift"], errors="coerce")\n    for column in ("net_cffoa", "net_profit", "total_assets"):\n        frame[column] = pd.to_numeric(frame[column], errors="coerce")\n    frame = frame.loc[frame["shift"].eq(0)].dropna(subset=["date", "instrument"])\n\n    ttm = frame.loc[\n        frame["category"].eq("ttm"),\n        ["date", "instrument", "report_date", "net_cffoa", "net_profit"],\n    ].copy()\n    ttm = (\n        ttm.sort_values(["instrument", "date", "report_date"])\n        .drop_duplicates(["date", "instrument"], keep="last")\n        .rename(columns={"date": "disclosure_date"})\n    )\n    assets = frame.loc[\n        frame["category"].eq("lf"),\n        ["date", "instrument", "total_assets"],\n    ].copy()\n    assets = (\n        assets.sort_values(["instrument", "date"])\n        .drop_duplicates(["date", "instrument"], keep="last")\n        .rename(columns={"date": "asset_disclosure_date"})\n    )\n    events = _group_asof(\n        ttm,\n        assets,\n        left_on="disclosure_date",\n        right_on="asset_disclosure_date",\n        right_columns=["total_assets"],\n    )\n    valid_assets = events["total_assets"].where(events["total_assets"] > 0)\n    events["cash_quality"] = (\n        events["net_cffoa"] - events["net_profit"]\n    ) / valid_assets\n    events = events.sort_values(["instrument", "disclosure_date", "report_date"])\n    events["factor_raw"] = events.groupby("instrument", sort=False)["cash_quality"].diff()\n    disclosure_values = events["disclosure_date"].to_numpy(dtype="datetime64[ns]")\n    calendar_values = calendar.to_numpy(dtype="datetime64[ns]")\n    positions = np.searchsorted(calendar_values, disclosure_values, side="right")\n    valid_position = positions < len(calendar_values)\n    events["effective_date"] = pd.NaT\n    events.loc[valid_position, "effective_date"] = calendar_values[positions[valid_position]]\n    events["factor_raw"] = events["factor_raw"].replace([np.inf, -np.inf], np.nan)\n    return (\n        events[\n            [\n                "instrument",\n                "disclosure_date",\n                "effective_date",\n                "report_date",\n                "cash_quality",\n                "factor_raw",\n            ]\n        ]\n        .dropna(subset=["effective_date"])\n        .sort_values(["instrument", "effective_date"])\n        .reset_index(drop=True)\n    )\n\n\ndef build_fr_001_factor(\n    financial: pd.DataFrame,\n    pool: pd.DataFrame,\n    trading_days: Iterable[object],\n    *,\n    start_date: object | None = None,\n    end_date: object | None = None,\n) -> pd.DataFrame:\n    """Return the exact ``date, instrument, factor`` research interface."""\n\n    _require_columns(pool, POOL_COLUMNS, "pool")\n    events = compute_fr_001_events(financial, trading_days)\n    panel = pool.loc[:, POOL_COLUMNS].copy()\n    panel["date"] = pd.to_datetime(panel["date"], errors="coerce").dt.normalize().astype("datetime64[ns]")\n    panel["instrument"] = panel["instrument"].astype(str)\n    panel = panel.dropna(subset=["date", "instrument"])\n    if start_date is not None:\n        panel = panel.loc[panel["date"] >= pd.Timestamp(start_date).normalize()]\n    if end_date is not None:\n        panel = panel.loc[panel["date"] <= pd.Timestamp(end_date).normalize()]\n    state = _group_asof(\n        panel,\n        events,\n        left_on="date",\n        right_on="effective_date",\n        right_columns=["factor_raw"],\n    )\n    daily_median = state.groupby("date", sort=False)["factor_raw"].transform("median")\n    state["factor_raw"] = state["factor_raw"].fillna(daily_median).fillna(0.0)\n    state["factor"] = (\n        state.groupby("date", sort=False)["factor_raw"]\n        .rank(pct=True, method="average")\n        .sub(0.5)\n        .mul(2.0)\n    )\n    if not np.isfinite(state["factor"]).all():\n        raise ValueError("FR-001 produced non-finite factor values")\n    return (\n        state.loc[:, OUTPUT_COLUMNS]\n        .drop_duplicates(["date", "instrument"], keep="last")\n        .sort_values(["date", "instrument"])\n        .reset_index(drop=True)\n    )\n\n\ndef compute_fr_001_events_from_panel(financial: pd.DataFrame) -> pd.DataFrame:\n    """Compute FR-001 events while preserving AIStudio\'s PIT effective date."""\n\n    required = (\n        "disclosure_date",\n        "effective_date",\n        "instrument",\n        "report_date",\n        "category",\n        "shift",\n        "net_cffoa",\n        "net_profit",\n        "total_assets",\n    )\n    _require_columns(financial, required, "financial")\n    frame = financial.loc[:, required].copy()\n    for column in ("disclosure_date", "effective_date", "report_date"):\n        frame[column] = pd.to_datetime(frame[column], errors="coerce").dt.normalize()\n    frame["instrument"] = frame["instrument"].astype(str)\n    frame["category"] = frame["category"].astype(str).str.lower()\n    frame["shift"] = pd.to_numeric(frame["shift"], errors="coerce")\n    for column in ("net_cffoa", "net_profit", "total_assets"):\n        frame[column] = pd.to_numeric(frame[column], errors="coerce")\n    frame = frame.loc[frame["shift"].eq(0)].dropna(\n        subset=["disclosure_date", "effective_date", "instrument"]\n    )\n    ttm = (\n        frame.loc[\n            frame["category"].eq("ttm"),\n            [\n                "disclosure_date",\n                "effective_date",\n                "instrument",\n                "report_date",\n                "net_cffoa",\n                "net_profit",\n            ],\n        ]\n        .sort_values(["instrument", "disclosure_date", "report_date"])\n        .drop_duplicates(["disclosure_date", "instrument"], keep="last")\n    )\n    assets = (\n        frame.loc[\n            frame["category"].eq("lf"),\n            ["disclosure_date", "instrument", "total_assets"],\n        ]\n        .sort_values(["instrument", "disclosure_date"])\n        .drop_duplicates(["disclosure_date", "instrument"], keep="last")\n        .rename(columns={"disclosure_date": "asset_disclosure_date"})\n    )\n    events = _group_asof(\n        ttm,\n        assets,\n        left_on="disclosure_date",\n        right_on="asset_disclosure_date",\n        right_columns=["total_assets"],\n    )\n    valid_assets = events["total_assets"].where(events["total_assets"] > 0)\n    events["cash_quality"] = (\n        events["net_cffoa"] - events["net_profit"]\n    ) / valid_assets\n    events = events.sort_values(["instrument", "disclosure_date", "report_date"])\n    events["factor_raw"] = events.groupby("instrument", sort=False)["cash_quality"].diff()\n    events["factor_raw"] = events["factor_raw"].replace([np.inf, -np.inf], np.nan)\n    return events[\n        [\n            "instrument",\n            "disclosure_date",\n            "effective_date",\n            "report_date",\n            "cash_quality",\n            "factor_raw",\n        ]\n    ].sort_values(["instrument", "effective_date"]).reset_index(drop=True)\n\n\ndef build_fr_001_factor_from_panel(\n    financial: pd.DataFrame,\n    pool: pd.DataFrame,\n) -> pd.DataFrame:\n    """Build FR-001 from the frozen AIStudio PIT event panel."""\n\n    _require_columns(pool, POOL_COLUMNS, "pool")\n    events = compute_fr_001_events_from_panel(financial)\n    panel = pool.loc[:, POOL_COLUMNS].copy()\n    panel["date"] = pd.to_datetime(panel["date"], errors="coerce").dt.normalize().astype("datetime64[ns]")\n    panel["instrument"] = panel["instrument"].astype(str)\n    state = _group_asof(\n        panel,\n        events,\n        left_on="date",\n        right_on="effective_date",\n        right_columns=["factor_raw"],\n    )\n    median = state.groupby("date", sort=False)["factor_raw"].transform("median")\n    state["factor_raw"] = state["factor_raw"].fillna(median).fillna(0.0)\n    state["factor"] = (\n        state.groupby("date", sort=False)["factor_raw"]\n        .rank(pct=True, method="average")\n        .sub(0.5)\n        .mul(2.0)\n    )\n    if not np.isfinite(state["factor"]).all():\n        raise ValueError("FR-001 produced non-finite factor values")\n    return state.loc[:, OUTPUT_COLUMNS].sort_values(\n        ["date", "instrument"]\n    ).reset_index(drop=True)\n', 'bigalpha2026.candidates.fr.fr_002': '"""FR-002: newly disclosed asset-efficiency improvement."""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Iterable\n\nimport numpy as np\nimport pandas as pd\n\nFINANCIAL_COLUMNS = (\n    "date",\n    "instrument",\n    "category",\n    "shift",\n    "report_date",\n    "operating_revenue",\n    "net_profit",\n    "total_assets",\n)\nPOOL_COLUMNS = ("date", "instrument")\nOUTPUT_COLUMNS = ("date", "instrument", "factor")\nCOMPONENT_COLUMNS = ("turnover_change", "roa_change")\n\n\ndef _require_columns(frame: pd.DataFrame, columns: Iterable[str], name: str) -> None:\n    missing = sorted(set(columns).difference(frame.columns))\n    if missing:\n        raise ValueError(f"{name} is missing required columns: {missing}")\n\n\ndef _group_asof(\n    left: pd.DataFrame,\n    right: pd.DataFrame,\n    *,\n    left_on: str,\n    right_on: str,\n    right_columns: list[str],\n) -> pd.DataFrame:\n    left_frame = left.copy()\n    left_frame[left_on] = pd.to_datetime(\n        left_frame[left_on], errors="coerce"\n    ).astype("datetime64[ns]")\n    right_frame = right.copy()\n    right_frame[right_on] = pd.to_datetime(\n        right_frame[right_on], errors="coerce"\n    ).astype("datetime64[ns]")\n    left_frame["_row_order"] = np.arange(len(left_frame))\n    right_groups = {\n        instrument: block.sort_values(right_on)\n        for instrument, block in right_frame.groupby("instrument", sort=False)\n    }\n    pieces: list[pd.DataFrame] = []\n    for instrument, left_block in left_frame.groupby("instrument", sort=False):\n        ordered = left_block.sort_values(left_on)\n        right_block = right_groups.get(instrument)\n        if right_block is None or right_block.empty:\n            for column in [right_on, *right_columns]:\n                ordered[column] = pd.NaT if column == right_on else np.nan\n            pieces.append(ordered)\n            continue\n        pieces.append(\n            pd.merge_asof(\n                ordered,\n                right_block[[right_on, *right_columns]].sort_values(right_on),\n                left_on=left_on,\n                right_on=right_on,\n                direction="backward",\n                allow_exact_matches=True,\n            )\n        )\n    if not pieces:\n        return left_frame.drop(columns="_row_order")\n    return (\n        pd.concat(pieces, ignore_index=True)\n        .sort_values("_row_order")\n        .drop(columns="_row_order")\n        .reset_index(drop=True)\n    )\n\n\ndef compute_fr_002_events(\n    financial: pd.DataFrame,\n    trading_days: Iterable[object],\n) -> pd.DataFrame:\n    """Build PIT changes in TTM revenue/assets and profit/assets."""\n\n    if "date" not in financial.columns and "disclosure_date" in financial.columns:\n        financial = financial.rename(columns={"disclosure_date": "date"})\n    _require_columns(financial, FINANCIAL_COLUMNS, "financial")\n    calendar = pd.DatetimeIndex(pd.to_datetime(list(trading_days), errors="coerce"))\n    calendar = calendar.dropna().normalize().unique().sort_values()\n    if len(calendar) == 0:\n        raise ValueError("trading_days must contain at least one valid date")\n\n    frame = financial.loc[:, FINANCIAL_COLUMNS].copy()\n    frame["date"] = pd.to_datetime(frame["date"], errors="coerce").dt.normalize()\n    frame["report_date"] = pd.to_datetime(\n        frame["report_date"],\n        errors="coerce",\n    ).dt.normalize()\n    frame["instrument"] = frame["instrument"].astype(str)\n    frame["category"] = frame["category"].astype(str).str.lower()\n    frame["shift"] = pd.to_numeric(frame["shift"], errors="coerce")\n    for column in ("operating_revenue", "net_profit", "total_assets"):\n        frame[column] = pd.to_numeric(frame[column], errors="coerce")\n    frame = frame.loc[frame["shift"].eq(0)].dropna(subset=["date", "instrument"])\n\n    ttm = frame.loc[\n        frame["category"].eq("ttm"),\n        ["date", "instrument", "report_date", "operating_revenue", "net_profit"],\n    ].copy()\n    ttm = (\n        ttm.sort_values(["instrument", "date", "report_date"])\n        .drop_duplicates(["date", "instrument"], keep="last")\n        .rename(columns={"date": "disclosure_date"})\n    )\n    assets = frame.loc[\n        frame["category"].eq("lf"),\n        ["date", "instrument", "total_assets"],\n    ].copy()\n    assets = (\n        assets.sort_values(["instrument", "date"])\n        .drop_duplicates(["date", "instrument"], keep="last")\n        .rename(columns={"date": "asset_disclosure_date"})\n    )\n    events = _group_asof(\n        ttm,\n        assets,\n        left_on="disclosure_date",\n        right_on="asset_disclosure_date",\n        right_columns=["total_assets"],\n    )\n    valid_assets = events["total_assets"].where(events["total_assets"] > 0)\n    events["asset_turnover"] = events["operating_revenue"] / valid_assets\n    events["roa_proxy"] = events["net_profit"] / valid_assets\n    events = events.sort_values(["instrument", "disclosure_date", "report_date"])\n    events["turnover_change"] = events.groupby("instrument", sort=False)[\n        "asset_turnover"\n    ].diff()\n    events["roa_change"] = events.groupby("instrument", sort=False)["roa_proxy"].diff()\n\n    disclosure_values = events["disclosure_date"].to_numpy(dtype="datetime64[ns]")\n    calendar_values = calendar.to_numpy(dtype="datetime64[ns]")\n    positions = np.searchsorted(calendar_values, disclosure_values, side="right")\n    valid_position = positions < len(calendar_values)\n    events["effective_date"] = pd.NaT\n    events.loc[valid_position, "effective_date"] = calendar_values[positions[valid_position]]\n    events[list(COMPONENT_COLUMNS)] = events[list(COMPONENT_COLUMNS)].replace(\n        [np.inf, -np.inf],\n        np.nan,\n    )\n    return (\n        events[\n            [\n                "instrument",\n                "disclosure_date",\n                "effective_date",\n                "report_date",\n                *COMPONENT_COLUMNS,\n            ]\n        ]\n        .dropna(subset=["effective_date"])\n        .sort_values(["instrument", "effective_date"])\n        .reset_index(drop=True)\n    )\n\n\ndef build_fr_002_factor(\n    financial: pd.DataFrame,\n    pool: pd.DataFrame,\n    trading_days: Iterable[object],\n    *,\n    start_date: object | None = None,\n    end_date: object | None = None,\n) -> pd.DataFrame:\n    """Return the exact ``date, instrument, factor`` research interface."""\n\n    _require_columns(pool, POOL_COLUMNS, "pool")\n    events = compute_fr_002_events(financial, trading_days)\n    panel = pool.loc[:, POOL_COLUMNS].copy()\n    panel["date"] = pd.to_datetime(panel["date"], errors="coerce").dt.normalize().astype("datetime64[ns]")\n    panel["instrument"] = panel["instrument"].astype(str)\n    if start_date is not None:\n        panel = panel.loc[panel["date"] >= pd.Timestamp(start_date).normalize()]\n    if end_date is not None:\n        panel = panel.loc[panel["date"] <= pd.Timestamp(end_date).normalize()]\n    state = _group_asof(\n        panel,\n        events,\n        left_on="date",\n        right_on="effective_date",\n        right_columns=list(COMPONENT_COLUMNS),\n    )\n    ranks: list[pd.Series] = []\n    for column in COMPONENT_COLUMNS:\n        values = pd.to_numeric(state[column], errors="coerce")\n        median = values.groupby(state["date"], sort=False).transform("median")\n        values = values.fillna(median)\n        ranks.append(\n            values.groupby(state["date"], sort=False)\n            .rank(pct=True, method="average")\n            .fillna(0.5)\n        )\n    state["factor_raw"] = pd.concat(ranks, axis=1).mean(axis=1)\n    state["factor"] = (\n        state.groupby("date", sort=False)["factor_raw"]\n        .rank(pct=True, method="average")\n        .sub(0.5)\n        .mul(2.0)\n    )\n    if not np.isfinite(state["factor"]).all():\n        raise ValueError("FR-002 produced non-finite factor values")\n    return (\n        state.loc[:, OUTPUT_COLUMNS]\n        .drop_duplicates(["date", "instrument"], keep="last")\n        .sort_values(["date", "instrument"])\n        .reset_index(drop=True)\n    )\n\n\ndef compute_fr_002_events_from_panel(financial: pd.DataFrame) -> pd.DataFrame:\n    """Compute FR-002 events while preserving AIStudio\'s PIT effective date."""\n\n    required = (\n        "disclosure_date",\n        "effective_date",\n        "instrument",\n        "report_date",\n        "category",\n        "shift",\n        "operating_revenue",\n        "net_profit",\n        "total_assets",\n    )\n    _require_columns(financial, required, "financial")\n    frame = financial.loc[:, required].copy()\n    for column in ("disclosure_date", "effective_date", "report_date"):\n        frame[column] = pd.to_datetime(frame[column], errors="coerce").dt.normalize()\n    frame["instrument"] = frame["instrument"].astype(str)\n    frame["category"] = frame["category"].astype(str).str.lower()\n    frame["shift"] = pd.to_numeric(frame["shift"], errors="coerce")\n    for column in ("operating_revenue", "net_profit", "total_assets"):\n        frame[column] = pd.to_numeric(frame[column], errors="coerce")\n    frame = frame.loc[frame["shift"].eq(0)].dropna(\n        subset=["disclosure_date", "effective_date", "instrument"]\n    )\n    ttm = (\n        frame.loc[\n            frame["category"].eq("ttm"),\n            [\n                "disclosure_date",\n                "effective_date",\n                "instrument",\n                "report_date",\n                "operating_revenue",\n                "net_profit",\n            ],\n        ]\n        .sort_values(["instrument", "disclosure_date", "report_date"])\n        .drop_duplicates(["disclosure_date", "instrument"], keep="last")\n    )\n    assets = (\n        frame.loc[\n            frame["category"].eq("lf"),\n            ["disclosure_date", "instrument", "total_assets"],\n        ]\n        .sort_values(["instrument", "disclosure_date"])\n        .drop_duplicates(["disclosure_date", "instrument"], keep="last")\n        .rename(columns={"disclosure_date": "asset_disclosure_date"})\n    )\n    events = _group_asof(\n        ttm,\n        assets,\n        left_on="disclosure_date",\n        right_on="asset_disclosure_date",\n        right_columns=["total_assets"],\n    )\n    valid_assets = events["total_assets"].where(events["total_assets"] > 0)\n    events["asset_turnover"] = events["operating_revenue"] / valid_assets\n    events["roa_proxy"] = events["net_profit"] / valid_assets\n    events = events.sort_values(["instrument", "disclosure_date", "report_date"])\n    events["turnover_change"] = events.groupby("instrument", sort=False)[\n        "asset_turnover"\n    ].diff()\n    events["roa_change"] = events.groupby("instrument", sort=False)["roa_proxy"].diff()\n    events[list(COMPONENT_COLUMNS)] = events[list(COMPONENT_COLUMNS)].replace(\n        [np.inf, -np.inf],\n        np.nan,\n    )\n    return events[\n        [\n            "instrument",\n            "disclosure_date",\n            "effective_date",\n            "report_date",\n            *COMPONENT_COLUMNS,\n        ]\n    ].sort_values(["instrument", "effective_date"]).reset_index(drop=True)\n\n\ndef build_fr_002_factor_from_panel(\n    financial: pd.DataFrame,\n    pool: pd.DataFrame,\n) -> pd.DataFrame:\n    """Build FR-002 from the frozen AIStudio PIT event panel."""\n\n    _require_columns(pool, POOL_COLUMNS, "pool")\n    events = compute_fr_002_events_from_panel(financial)\n    panel = pool.loc[:, POOL_COLUMNS].copy()\n    panel["date"] = pd.to_datetime(panel["date"], errors="coerce").dt.normalize().astype("datetime64[ns]")\n    panel["instrument"] = panel["instrument"].astype(str)\n    state = _group_asof(\n        panel,\n        events,\n        left_on="date",\n        right_on="effective_date",\n        right_columns=list(COMPONENT_COLUMNS),\n    )\n    ranks: list[pd.Series] = []\n    for column in COMPONENT_COLUMNS:\n        values = pd.to_numeric(state[column], errors="coerce")\n        values = values.fillna(values.groupby(state["date"], sort=False).transform("median"))\n        ranks.append(\n            values.groupby(state["date"], sort=False)\n            .rank(pct=True, method="average")\n            .fillna(0.5)\n        )\n    state["factor_raw"] = pd.concat(ranks, axis=1).mean(axis=1)\n    state["factor"] = (\n        state.groupby("date", sort=False)["factor_raw"]\n        .rank(pct=True, method="average")\n        .sub(0.5)\n        .mul(2.0)\n    )\n    if not np.isfinite(state["factor"]).all():\n        raise ValueError("FR-002 produced non-finite factor values")\n    return state.loc[:, OUTPUT_COLUMNS].sort_values(\n        ["date", "instrument"]\n    ).reset_index(drop=True)\n', 'bigalpha2026.candidates.fr.fr_004': '"""FR-004: OAP-inspired PIT TTM revenue-growth surprise."""\n\nfrom __future__ import annotations\n\nimport pandas as pd\n\nfrom ._common import build_event_factor, year_over_year_events\n\n\ndef compute_fr_004_events(financial: pd.DataFrame) -> pd.DataFrame:\n    """Compute year-over-year TTM revenue growth known at each disclosure.\n\n    The OAP original uses quarterly revenue per share and a historical surprise\n    normalization. Those inputs are unavailable in the current contract, so\n    this is explicitly an adapted A-share mechanism, not an exact replication.\n    """\n\n    events = year_over_year_events(\n        financial,\n        category="ttm",\n        value_column="operating_revenue",\n        output_column="factor_raw",\n        sign=1.0,\n    )\n    return events[\n        [\n            "instrument",\n            "disclosure_date",\n            "effective_date",\n            "report_date",\n            "factor_raw",\n        ]\n    ]\n\n\ndef build_fr_004_factor(\n    financial: pd.DataFrame,\n    pool: pd.DataFrame,\n) -> pd.DataFrame:\n    events = compute_fr_004_events(financial)\n    return build_event_factor(\n        events,\n        pool,\n        value_column="factor_raw",\n        candidate_id="FR-004",\n    )\n', 'bigalpha2026.candidates.fr.fr_005': '"""FR-005: OAP operating cash flow-to-market (cfp)."""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom ._common import (\n    group_asof,\n    prepare_event_panel,\n    prepare_pool,\n    rank_state,\n    require_columns,\n)\n\n\ndef compute_fr_005_events(financial: pd.DataFrame) -> pd.DataFrame:\n    """Keep each newly disclosed TTM operating cash-flow state."""\n\n    events = prepare_event_panel(\n        financial,\n        category="ttm",\n        value_column="net_cffoa",\n    ).rename(columns={"net_cffoa": "operating_cash_flow"})\n    events["operating_cash_flow"] = pd.to_numeric(\n        events["operating_cash_flow"],\n        errors="coerce",\n    ).replace([np.inf, -np.inf], np.nan)\n    return (\n        events[\n            [\n                "instrument",\n                "disclosure_date",\n                "effective_date",\n                "report_date",\n                "operating_cash_flow",\n            ]\n        ]\n        .sort_values(\n            ["instrument", "effective_date", "report_date", "disclosure_date"]\n        )\n        .drop_duplicates(["instrument", "effective_date"], keep="last")\n        .reset_index(drop=True)\n    )\n\n\ndef build_fr_005_factor(\n    financial: pd.DataFrame,\n    exposures: pd.DataFrame,\n    pool: pd.DataFrame,\n) -> pd.DataFrame:\n    """Divide the PIT cash-flow state by same-day float market capitalization."""\n\n    events = compute_fr_005_events(financial)\n    panel = prepare_pool(pool)\n    state = group_asof(\n        panel,\n        events,\n        left_on="date",\n        right_on="effective_date",\n        right_columns=["operating_cash_flow"],\n    )\n    required = ("date", "instrument", "float_market_cap")\n    require_columns(exposures, required, "exposures")\n    market = exposures.loc[:, required].copy()\n    market["date"] = pd.to_datetime(market["date"], errors="coerce").dt.normalize()\n    market["instrument"] = market["instrument"].astype(str)\n    market["float_market_cap"] = pd.to_numeric(\n        market["float_market_cap"],\n        errors="coerce",\n    )\n    if market.duplicated(["date", "instrument"]).any():\n        raise ValueError("exposures contains duplicate date-instrument keys")\n    state = state.merge(\n        market,\n        on=["date", "instrument"],\n        how="left",\n        validate="one_to_one",\n    )\n    valid_market_cap = state["float_market_cap"].where(\n        state["float_market_cap"] > 0\n    )\n    state["factor_raw"] = state["operating_cash_flow"] / valid_market_cap\n    return rank_state(state, candidate_id="FR-005")\n', 'bigalpha2026.candidates.fr.fr_006': '"""FR-006: OAP-inspired PIT TTM earnings-growth surprise."""\n\nfrom __future__ import annotations\n\nimport pandas as pd\n\nfrom ._common import build_event_factor, year_over_year_events\n\n\ndef compute_fr_006_events(financial: pd.DataFrame) -> pd.DataFrame:\n    """Compute the disclosed year-over-year change in TTM net profit."""\n\n    events = year_over_year_events(\n        financial,\n        category="ttm",\n        value_column="net_profit",\n        output_column="factor_raw",\n        sign=1.0,\n    )\n    return events[\n        [\n            "instrument",\n            "disclosure_date",\n            "effective_date",\n            "report_date",\n            "factor_raw",\n        ]\n    ]\n\n\ndef build_fr_006_factor(\n    financial: pd.DataFrame,\n    pool: pd.DataFrame,\n) -> pd.DataFrame:\n    return build_event_factor(\n        compute_fr_006_events(financial),\n        pool,\n        value_column="factor_raw",\n        candidate_id="FR-006",\n    )\n', 'bigalpha2026.candidates.fr.fr_010': '"""FR-010: abnormal-accruals proxy from the available PIT contract."""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom ._common import build_event_factor, group_asof, prepare_event_panel\n\n\ndef compute_fr_010_events(financial: pd.DataFrame) -> pd.DataFrame:\n    """Use negative cash accruals scaled by the latest known total assets.\n\n    PPE and the original industry regression are unavailable, so this is a\n    documented proxy rather than a reproduction of the OAP signal.\n    """\n\n    profit = prepare_event_panel(\n        financial,\n        category="ttm",\n        value_column="net_profit",\n    )\n    cash = prepare_event_panel(\n        financial,\n        category="ttm",\n        value_column="net_cffoa",\n    )\n    ttm = profit.merge(\n        cash[\n            [\n                "instrument",\n                "disclosure_date",\n                "effective_date",\n                "report_date",\n                "net_cffoa",\n            ]\n        ],\n        on=["instrument", "disclosure_date", "effective_date", "report_date"],\n        how="inner",\n        validate="one_to_one",\n    )\n    assets = prepare_event_panel(\n        financial,\n        category="lf",\n        value_column="total_assets",\n    ).rename(columns={"effective_date": "asset_effective_date"})\n    events = group_asof(\n        ttm,\n        assets,\n        left_on="effective_date",\n        right_on="asset_effective_date",\n        right_columns=["total_assets"],\n    )\n    denominator = events["total_assets"].where(events["total_assets"].abs() > 1e-12)\n    events["factor_raw"] = -(\n        (events["net_profit"] - events["net_cffoa"]) / denominator\n    )\n    events["factor_raw"] = events["factor_raw"].replace(\n        [np.inf, -np.inf],\n        np.nan,\n    )\n    return events\n\n\ndef build_fr_010_factor(\n    financial: pd.DataFrame,\n    pool: pd.DataFrame,\n) -> pd.DataFrame:\n    return build_event_factor(\n        compute_fr_010_events(financial),\n        pool,\n        value_column="factor_raw",\n        candidate_id="FR-010",\n    )\n', 'bigalpha2026.candidates.fr.fr_011': '"""FR-011: PIT assets-to-market."""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom ._common import group_asof, prepare_event_panel, prepare_pool, rank_state\n\n\ndef build_fr_011_factor(\n    financial: pd.DataFrame,\n    exposures: pd.DataFrame,\n    pool: pd.DataFrame,\n) -> pd.DataFrame:\n    assets = prepare_event_panel(\n        financial,\n        category="lf",\n        value_column="total_assets",\n    )\n    panel = prepare_pool(pool)\n    state = group_asof(\n        panel,\n        assets,\n        left_on="date",\n        right_on="effective_date",\n        right_columns=["total_assets"],\n    )\n    market_cap = exposures[\n        ["date", "instrument", "float_market_cap"]\n    ].copy()\n    market_cap["date"] = pd.to_datetime(\n        market_cap["date"],\n        errors="coerce",\n    ).dt.normalize()\n    market_cap["instrument"] = market_cap["instrument"].astype(str)\n    if market_cap.duplicated(["date", "instrument"]).any():\n        raise ValueError("exposures contains duplicate date-instrument keys")\n    state = state.merge(\n        market_cap,\n        on=["date", "instrument"],\n        how="left",\n        validate="one_to_one",\n    )\n    denominator = state["float_market_cap"].where(state["float_market_cap"] > 0)\n    state["factor_raw"] = state["total_assets"] / denominator\n    state["factor_raw"] = state["factor_raw"].replace(\n        [np.inf, -np.inf],\n        np.nan,\n    )\n    return rank_state(state, candidate_id="FR-011")\n', 'bigalpha2026.candidates.fr.fr_012': '"""FR-012: revenue-confirmed standardized earnings surprise."""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom ._common import build_event_factor, year_over_year_events\n\nEVENT_KEYS = (\n    "instrument",\n    "disclosure_date",\n    "effective_date",\n    "report_date",\n)\n\n\ndef _prior_standardized(\n    frame: pd.DataFrame,\n    column: str,\n    *,\n    history_window: int,\n    min_periods: int,\n) -> pd.Series:\n    grouped = frame.groupby("instrument", sort=False)[column]\n    prior = grouped.shift(1)\n    rolling = prior.groupby(frame["instrument"], sort=False).rolling(\n        history_window,\n        min_periods=min_periods,\n    )\n    mean = rolling.mean().reset_index(level=0, drop=True).sort_index()\n    std = rolling.std().reset_index(level=0, drop=True).sort_index()\n    return ((frame[column] - mean) / std.where(std > 1e-12)).clip(-5.0, 5.0)\n\n\ndef compute_fr_012_events(\n    financial: pd.DataFrame,\n    *,\n    history_window: int = 8,\n    min_periods: int = 4,\n) -> pd.DataFrame:\n    """Keep only earnings surprises confirmed by same-signed revenue news."""\n\n    if history_window < 2:\n        raise ValueError("history_window must be at least 2")\n    if min_periods < 2 or min_periods > history_window:\n        raise ValueError("min_periods must be between 2 and history_window")\n\n    earnings = year_over_year_events(\n        financial,\n        category="ttm",\n        value_column="net_profit",\n        output_column="earnings_growth",\n        sign=1.0,\n    )\n    revenue = year_over_year_events(\n        financial,\n        category="ttm",\n        value_column="operating_revenue",\n        output_column="revenue_growth",\n        sign=1.0,\n    )\n    events = earnings[[*EVENT_KEYS, "earnings_growth"]].merge(\n        revenue[[*EVENT_KEYS, "revenue_growth"]],\n        on=list(EVENT_KEYS),\n        how="inner",\n        validate="one_to_one",\n    )\n    events = events.sort_values(\n        ["instrument", "effective_date", "report_date", "disclosure_date"]\n    ).reset_index(drop=True)\n    events["earnings_surprise"] = _prior_standardized(\n        events,\n        "earnings_growth",\n        history_window=history_window,\n        min_periods=min_periods,\n    )\n    events["revenue_surprise"] = _prior_standardized(\n        events,\n        "revenue_growth",\n        history_window=history_window,\n        min_periods=min_periods,\n    )\n\n    earnings_surprise = events["earnings_surprise"]\n    revenue_surprise = events["revenue_surprise"]\n    available = earnings_surprise.notna() & revenue_surprise.notna()\n    confirmed = available & (np.sign(earnings_surprise) == np.sign(revenue_surprise))\n    events["factor_raw"] = 0.0\n    events.loc[confirmed, "factor_raw"] = np.sign(earnings_surprise.loc[confirmed]) * np.sqrt(\n        earnings_surprise.loc[confirmed].abs() * revenue_surprise.loc[confirmed].abs()\n    )\n    events.loc[~available, "factor_raw"] = np.nan\n    return events[[*EVENT_KEYS, "factor_raw"]]\n\n\ndef build_fr_012_factor(\n    financial: pd.DataFrame,\n    pool: pd.DataFrame,\n) -> pd.DataFrame:\n    return build_event_factor(\n        compute_fr_012_events(financial),\n        pool,\n        value_column="factor_raw",\n        candidate_id="FR-012",\n    )\n', 'bigalpha2026.candidates.fr.fr_013': '"""FR-013: financial-report timing surprise."""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom ._common import group_asof, prepare_pool, rank_state, require_columns\n\nEVENT_COLUMNS = (\n    "disclosure_date",\n    "effective_date",\n    "instrument",\n    "report_date",\n    "category",\n    "shift",\n)\n\n\ndef compute_fr_013_events(\n    financial: pd.DataFrame,\n    *,\n    history_window: int = 3,\n    min_periods: int = 2,\n) -> pd.DataFrame:\n    """Compare the disclosure lag with strictly prior same-quarter lags."""\n\n    if history_window < 2:\n        raise ValueError("history_window must be at least 2")\n    if min_periods < 2 or min_periods > history_window:\n        raise ValueError("min_periods must be between 2 and history_window")\n    require_columns(financial, EVENT_COLUMNS, "financial")\n    events = financial.loc[:, EVENT_COLUMNS].copy()\n    for column in ("disclosure_date", "effective_date", "report_date"):\n        events[column] = pd.to_datetime(events[column], errors="coerce").dt.normalize()\n    events["instrument"] = events["instrument"].astype(str)\n    events["category"] = events["category"].astype(str).str.lower()\n    events["shift"] = pd.to_numeric(events["shift"], errors="coerce")\n    events = events.loc[events["category"].eq("ttm") & events["shift"].eq(0)]\n    events = (\n        events.dropna(\n            subset=[\n                "disclosure_date",\n                "effective_date",\n                "instrument",\n                "report_date",\n            ]\n        )\n        .sort_values(["instrument", "report_date", "disclosure_date"])\n        .drop_duplicates(["instrument", "report_date"], keep="first")\n        .reset_index(drop=True)\n    )\n    events["fiscal_quarter"] = events["report_date"].dt.quarter\n    events["delay_days"] = (\n        events["disclosure_date"] - events["report_date"]\n    ).dt.days.astype(float)\n    prior_median = events.groupby(\n        ["instrument", "fiscal_quarter"],\n        sort=False,\n    )["delay_days"].transform(\n        lambda values: values.shift(1).rolling(\n            history_window,\n            min_periods=min_periods,\n        ).median()\n    )\n    events["factor_raw"] = -(events["delay_days"] - prior_median)\n    events["factor_raw"] = events["factor_raw"].replace(\n        [np.inf, -np.inf],\n        np.nan,\n    )\n    return events[\n        [\n            "instrument",\n            "disclosure_date",\n            "effective_date",\n            "report_date",\n            "factor_raw",\n        ]\n    ]\n\n\ndef build_fr_013_factor(\n    financial: pd.DataFrame,\n    pool: pd.DataFrame,\n    *,\n    active_days: int = 20,\n) -> pd.DataFrame:\n    """Forward-fill each timing surprise for at most ``active_days`` sessions."""\n\n    if active_days < 1:\n        raise ValueError("active_days must be positive")\n    panel = prepare_pool(pool)\n    state = group_asof(\n        panel,\n        compute_fr_013_events(financial),\n        left_on="date",\n        right_on="effective_date",\n        right_columns=["factor_raw"],\n    )\n    state["event_age"] = np.nan\n    active = state["effective_date"].notna()\n    state.loc[active, "event_age"] = (\n        state.loc[active]\n        .groupby(["instrument", "effective_date"], sort=False)\n        .cumcount()\n        .astype(float)\n    )\n    state.loc[state["event_age"].ge(active_days), "factor_raw"] = np.nan\n    return rank_state(state, candidate_id="FR-013")\n', 'bigalpha2026.candidates.fr.fr_014': '"""FR-014: point-in-time earnings yield."""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom ._common import (\n    group_asof,\n    prepare_event_panel,\n    prepare_pool,\n    rank_state,\n    require_columns,\n)\n\n\ndef compute_fr_014_events(financial: pd.DataFrame) -> pd.DataFrame:\n    """Keep each newly disclosed TTM earnings state."""\n\n    events = prepare_event_panel(\n        financial,\n        category="ttm",\n        value_column="net_profit",\n    ).rename(columns={"net_profit": "earnings"})\n    events["earnings"] = pd.to_numeric(\n        events["earnings"],\n        errors="coerce",\n    ).replace([np.inf, -np.inf], np.nan)\n    return (\n        events[\n            [\n                "instrument",\n                "disclosure_date",\n                "effective_date",\n                "report_date",\n                "earnings",\n            ]\n        ]\n        .sort_values(\n            ["instrument", "effective_date", "report_date", "disclosure_date"]\n        )\n        .drop_duplicates(["instrument", "effective_date"], keep="last")\n        .reset_index(drop=True)\n    )\n\n\ndef build_fr_014_factor(\n    financial: pd.DataFrame,\n    exposures: pd.DataFrame,\n    pool: pd.DataFrame,\n) -> pd.DataFrame:\n    """Divide PIT TTM earnings by same-day float market capitalization."""\n\n    panel = prepare_pool(pool)\n    state = group_asof(\n        panel,\n        compute_fr_014_events(financial),\n        left_on="date",\n        right_on="effective_date",\n        right_columns=["earnings"],\n    )\n    required = ("date", "instrument", "float_market_cap")\n    require_columns(exposures, required, "exposures")\n    market = exposures.loc[:, required].copy()\n    market["date"] = pd.to_datetime(market["date"], errors="coerce").dt.normalize()\n    market["instrument"] = market["instrument"].astype(str)\n    market["float_market_cap"] = pd.to_numeric(\n        market["float_market_cap"],\n        errors="coerce",\n    )\n    if market.duplicated(["date", "instrument"]).any():\n        raise ValueError("exposures contains duplicate date-instrument keys")\n    state = state.merge(\n        market,\n        on=["date", "instrument"],\n        how="left",\n        validate="one_to_one",\n    )\n    valid_market_cap = state["float_market_cap"].where(\n        state["float_market_cap"] > 0\n    )\n    state["factor_raw"] = state["earnings"] / valid_market_cap\n    return rank_state(state, candidate_id="FR-014")\n', 'bigalpha2026.candidates.fr.fr_015': '"""FR-015: point-in-time year-over-year net-margin improvement."""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom ._common import build_event_factor, prepare_event_panel\n\nEVENT_COLUMNS = (\n    "instrument",\n    "disclosure_date",\n    "effective_date",\n    "report_date",\n)\n\n\ndef compute_fr_015_events(financial: pd.DataFrame) -> pd.DataFrame:\n    """Compare the latest TTM net margin with the same fiscal quarter last year."""\n\n    earnings = prepare_event_panel(\n        financial,\n        category="ttm",\n        value_column="net_profit",\n    )\n    revenue = prepare_event_panel(\n        financial,\n        category="ttm",\n        value_column="operating_revenue",\n    )\n    events = earnings[[*EVENT_COLUMNS, "net_profit"]].merge(\n        revenue[[*EVENT_COLUMNS, "operating_revenue"]],\n        on=list(EVENT_COLUMNS),\n        how="inner",\n        validate="one_to_one",\n    )\n    revenue_value = pd.to_numeric(\n        events["operating_revenue"],\n        errors="coerce",\n    )\n    earnings_value = pd.to_numeric(events["net_profit"], errors="coerce")\n    events["net_margin"] = earnings_value / revenue_value.where(\n        revenue_value.abs() > 1e-12\n    )\n    events = events.sort_values(\n        ["instrument", "disclosure_date", "report_date"]\n    ).reset_index(drop=True)\n\n    pieces: list[pd.DataFrame] = []\n    for _, block in events.groupby("instrument", sort=False):\n        history: dict[pd.Timestamp, float] = {}\n        improvements: list[float] = []\n        for row in block.itertuples(index=False):\n            report_date = pd.Timestamp(row.report_date)\n            current = float(row.net_margin)\n            previous = history.get(report_date - pd.DateOffset(years=1))\n            if (\n                previous is None\n                or not np.isfinite(previous)\n                or not np.isfinite(current)\n            ):\n                improvements.append(np.nan)\n            else:\n                improvements.append(current - previous)\n            if np.isfinite(current):\n                history[report_date] = current\n        enriched = block.copy()\n        enriched["factor_raw"] = improvements\n        pieces.append(enriched)\n    if not pieces:\n        events["factor_raw"] = np.nan\n        return events[[*EVENT_COLUMNS, "factor_raw"]]\n    output = pd.concat(pieces, ignore_index=True)\n    output["factor_raw"] = pd.to_numeric(\n        output["factor_raw"],\n        errors="coerce",\n    ).replace([np.inf, -np.inf], np.nan)\n    return (\n        output.sort_values(\n            ["instrument", "effective_date", "report_date", "disclosure_date"]\n        )\n        .drop_duplicates(["instrument", "effective_date"], keep="last")\n        .loc[:, [*EVENT_COLUMNS, "factor_raw"]]\n        .reset_index(drop=True)\n    )\n\n\ndef build_fr_015_factor(\n    financial: pd.DataFrame,\n    pool: pd.DataFrame,\n) -> pd.DataFrame:\n    return build_event_factor(\n        compute_fr_015_events(financial),\n        pool,\n        value_column="factor_raw",\n        candidate_id="FR-015",\n    )\n', 'bigalpha2026.candidates.pv.pv_002': '"""PV-002: intraday absorption of overnight gaps."""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Iterable\n\nimport numpy as np\nimport pandas as pd\n\nBAR_COLUMNS = ("date", "instrument", "open", "close", "pre_close")\nPOOL_COLUMNS = ("date", "instrument")\nOUTPUT_COLUMNS = ("date", "instrument", "factor")\n\n\ndef _require_columns(frame: pd.DataFrame, columns: Iterable[str], name: str) -> None:\n    missing = sorted(set(columns).difference(frame.columns))\n    if missing:\n        raise ValueError(f"{name} is missing required columns: {missing}")\n\n\ndef compute_pv_002_daily(minute_bars: pd.DataFrame) -> pd.DataFrame:\n    """Measure signed absorption of the overnight gap by the daily close."""\n\n    _require_columns(minute_bars, BAR_COLUMNS, "minute_bars")\n    frame = minute_bars.loc[:, BAR_COLUMNS].copy()\n    frame["timestamp"] = pd.to_datetime(frame["date"], errors="coerce")\n    frame["date"] = frame["timestamp"].dt.normalize()\n    frame["instrument"] = frame["instrument"].astype(str)\n    for column in ("open", "close", "pre_close"):\n        frame[column] = pd.to_numeric(frame[column], errors="coerce")\n    frame = frame.dropna(subset=["timestamp", "instrument"]).sort_values(\n        ["instrument", "timestamp"]\n    )\n    daily = (\n        frame.groupby(["date", "instrument"], sort=False)\n        .agg(\n            open=("open", "first"),\n            close=("close", "last"),\n            pre_close=("pre_close", "first"),\n        )\n        .reset_index()\n    )\n    valid_pre_close = daily["pre_close"].where(daily["pre_close"] > 0)\n    valid_open = daily["open"].where(daily["open"] > 0)\n    daily["overnight_gap"] = daily["open"] / valid_pre_close - 1.0\n    daily["intraday_return"] = daily["close"] / valid_open - 1.0\n    opposite_move = (\n        -np.sign(daily["overnight_gap"]) * daily["intraday_return"]\n    ).clip(lower=0)\n    valid_observation = daily["overnight_gap"].notna() & daily["intraday_return"].notna()\n    absorbed_fraction = (\n        opposite_move / daily["overnight_gap"].abs().replace(0, np.nan)\n    ).clip(upper=1.0)\n    daily["factor_raw"] = -daily["overnight_gap"] * absorbed_fraction\n    daily.loc[\n        valid_observation & daily["overnight_gap"].eq(0),\n        "factor_raw",\n    ] = 0.0\n    daily["factor_raw"] = daily["factor_raw"].where(valid_observation)\n    daily["factor_raw"] = daily["factor_raw"].replace([np.inf, -np.inf], np.nan)\n    return daily.sort_values(["instrument", "date"]).reset_index(drop=True)\n\n\ndef build_pv_002_factor(\n    minute_bars: pd.DataFrame,\n    pool: pd.DataFrame,\n    *,\n    start_date: object | None = None,\n    end_date: object | None = None,\n) -> pd.DataFrame:\n    """Return the exact ``date, instrument, factor`` research interface."""\n\n    _require_columns(pool, POOL_COLUMNS, "pool")\n    daily = compute_pv_002_daily(minute_bars)\n    panel = pool.loc[:, POOL_COLUMNS].copy()\n    panel["date"] = pd.to_datetime(panel["date"], errors="coerce").dt.normalize().astype("datetime64[ns]")\n    panel["instrument"] = panel["instrument"].astype(str)\n    if start_date is not None:\n        panel = panel.loc[panel["date"] >= pd.Timestamp(start_date).normalize()]\n    if end_date is not None:\n        panel = panel.loc[panel["date"] <= pd.Timestamp(end_date).normalize()]\n    result = panel.merge(\n        daily[["date", "instrument", "factor_raw"]],\n        on=["date", "instrument"],\n        how="left",\n        validate="one_to_one",\n    )\n    daily_median = result.groupby("date", sort=False)["factor_raw"].transform("median")\n    result["factor_raw"] = result["factor_raw"].fillna(daily_median).fillna(0.0)\n    result["factor"] = (\n        result.groupby("date", sort=False)["factor_raw"]\n        .rank(pct=True, method="average")\n        .sub(0.5)\n        .mul(2.0)\n    )\n    if not np.isfinite(result["factor"]).all():\n        raise ValueError("PV-002 produced non-finite factor values")\n    return (\n        result.loc[:, OUTPUT_COLUMNS]\n        .drop_duplicates(["date", "instrument"], keep="last")\n        .sort_values(["date", "instrument"])\n        .reset_index(drop=True)\n    )\n', 'bigalpha2026.candidates.pv.pv_003': '"""PV-003: OAP MaxRet adapted to a daily A-share signal."""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom ._common import build_ranked_factor, prepare_daily, rolling_by_instrument\n\n\ndef compute_pv_003_daily(\n    daily_bars: pd.DataFrame,\n    *,\n    window: int = 21,\n    min_periods: int = 10,\n) -> pd.DataFrame:\n    """Use the negative trailing maximum daily return.\n\n    OAP signs ``MaxRet`` negatively. The current trading day is included because\n    the factor is available only after that day\'s close.\n    """\n\n    frame = prepare_daily(\n        daily_bars,\n        ("date", "instrument", "close", "pre_close"),\n    )\n    valid_pre_close = frame["pre_close"].where(frame["pre_close"] > 0)\n    frame["daily_return"] = frame["close"] / valid_pre_close - 1.0\n    frame["factor_raw"] = -rolling_by_instrument(\n        frame,\n        "daily_return",\n        window=window,\n        min_periods=min_periods,\n        method="max",\n    )\n    frame["factor_raw"] = frame["factor_raw"].replace([np.inf, -np.inf], np.nan)\n    return frame\n\n\ndef build_pv_003_factor(\n    daily_bars: pd.DataFrame,\n    pool: pd.DataFrame,\n    *,\n    start_date: object | None = None,\n    end_date: object | None = None,\n    window: int = 21,\n    min_periods: int = 10,\n) -> pd.DataFrame:\n    features = compute_pv_003_daily(\n        daily_bars,\n        window=window,\n        min_periods=min_periods,\n    )\n    return build_ranked_factor(\n        features,\n        pool,\n        candidate_id="PV-003",\n        start_date=start_date,\n        end_date=end_date,\n    )\n', 'bigalpha2026.candidates.pv.pv_004': '"""PV-004: OAP ReturnSkew adapted to a daily A-share signal."""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom ._common import build_ranked_factor, prepare_daily, rolling_by_instrument\n\n\ndef compute_pv_004_daily(\n    daily_bars: pd.DataFrame,\n    *,\n    window: int = 21,\n    min_periods: int = 10,\n) -> pd.DataFrame:\n    """Use negative trailing daily-return skewness, following OAP\'s sign."""\n\n    frame = prepare_daily(\n        daily_bars,\n        ("date", "instrument", "close", "pre_close"),\n    )\n    valid_pre_close = frame["pre_close"].where(frame["pre_close"] > 0)\n    frame["daily_return"] = frame["close"] / valid_pre_close - 1.0\n    frame["factor_raw"] = -rolling_by_instrument(\n        frame,\n        "daily_return",\n        window=window,\n        min_periods=min_periods,\n        method="skew",\n    )\n    frame["factor_raw"] = frame["factor_raw"].replace([np.inf, -np.inf], np.nan)\n    return frame\n\n\ndef build_pv_004_factor(\n    daily_bars: pd.DataFrame,\n    pool: pd.DataFrame,\n    *,\n    start_date: object | None = None,\n    end_date: object | None = None,\n    window: int = 21,\n    min_periods: int = 10,\n) -> pd.DataFrame:\n    features = compute_pv_004_daily(\n        daily_bars,\n        window=window,\n        min_periods=min_periods,\n    )\n    return build_ranked_factor(\n        features,\n        pool,\n        candidate_id="PV-004",\n        start_date=start_date,\n        end_date=end_date,\n    )\n', 'bigalpha2026.candidates.pv.pv_005': '"""PV-005: OAP Amihud illiquidity adapted to A-share amount."""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom ._common import build_ranked_factor, prepare_daily, rolling_by_instrument\n\n\ndef compute_pv_005_daily(\n    daily_bars: pd.DataFrame,\n    *,\n    window: int = 21,\n    min_periods: int = 10,\n) -> pd.DataFrame:\n    """Compute trailing mean absolute return per unit of trading amount."""\n\n    frame = prepare_daily(\n        daily_bars,\n        ("date", "instrument", "close", "pre_close", "amount"),\n    )\n    valid_pre_close = frame["pre_close"].where(frame["pre_close"] > 0)\n    valid_amount = frame["amount"].where(frame["amount"] > 0)\n    frame["daily_return"] = frame["close"] / valid_pre_close - 1.0\n    frame["daily_illiquidity"] = frame["daily_return"].abs() / valid_amount\n    frame["factor_raw"] = rolling_by_instrument(\n        frame,\n        "daily_illiquidity",\n        window=window,\n        min_periods=min_periods,\n        method="mean",\n    )\n    frame["factor_raw"] = frame["factor_raw"].replace([np.inf, -np.inf], np.nan)\n    return frame\n\n\ndef build_pv_005_factor(\n    daily_bars: pd.DataFrame,\n    pool: pd.DataFrame,\n    *,\n    start_date: object | None = None,\n    end_date: object | None = None,\n    window: int = 21,\n    min_periods: int = 10,\n) -> pd.DataFrame:\n    features = compute_pv_005_daily(\n        daily_bars,\n        window=window,\n        min_periods=min_periods,\n    )\n    return build_ranked_factor(\n        features,\n        pool,\n        candidate_id="PV-005",\n        start_date=start_date,\n        end_date=end_date,\n    )\n', 'bigalpha2026.candidates.pv.pv_008': '"""PV-008: closeness to the trailing 52-week high."""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom ._common import build_ranked_factor, prepare_daily, rolling_by_instrument\n\n\ndef compute_pv_008_daily(\n    daily_bars: pd.DataFrame,\n    *,\n    window: int = 252,\n    min_periods: int = 200,\n) -> pd.DataFrame:\n    frame = prepare_daily(daily_bars, ("date", "instrument", "close"))\n    trailing_high = rolling_by_instrument(\n        frame,\n        "close",\n        window=window,\n        min_periods=min_periods,\n        method="max",\n    )\n    frame["factor_raw"] = frame["close"] / trailing_high.where(trailing_high > 0)\n    frame["factor_raw"] = frame["factor_raw"].replace([np.inf, -np.inf], np.nan)\n    return frame\n\n\ndef build_pv_008_factor(\n    daily_bars: pd.DataFrame,\n    pool: pd.DataFrame,\n    *,\n    window: int = 252,\n    min_periods: int = 200,\n) -> pd.DataFrame:\n    return build_ranked_factor(\n        compute_pv_008_daily(\n            daily_bars,\n            window=window,\n            min_periods=min_periods,\n        ),\n        pool,\n        candidate_id="PV-008",\n    )\n', 'bigalpha2026.candidates.pv.pv_009': '"""PV-009: adapted market-price-delay score."""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom ._common import build_ranked_factor, prepare_daily\n\n\ndef compute_pv_009_daily(\n    daily_bars: pd.DataFrame,\n    *,\n    window: int = 252,\n    min_periods: int = 200,\n    lags: int = 4,\n) -> pd.DataFrame:\n    """Estimate delay from rolling current and lagged market correlations.\n\n    This is an A-share approximation to PriceDelayRsq. It uses the share of\n    squared return-market correlation carried by four lagged market returns.\n    The sign is negative because lower delay is the hypothesised good state.\n    """\n\n    frame = prepare_daily(\n        daily_bars,\n        ("date", "instrument", "close", "pre_close"),\n    )\n    frame["stock_return"] = (\n        frame["close"] / frame["pre_close"].where(frame["pre_close"] > 0) - 1.0\n    )\n    market = (\n        frame.groupby("date", sort=True)["stock_return"]\n        .mean()\n        .rename("market_return")\n    )\n    frame = frame.merge(market, on="date", how="left", validate="many_to_one")\n    squared_correlations: list[pd.Series] = []\n    for lag in range(lags + 1):\n        lagged_market = market.shift(lag).rename(f"market_lag_{lag}")\n        frame = frame.merge(\n            lagged_market,\n            on="date",\n            how="left",\n            validate="many_to_one",\n        )\n        rolling_corr = pd.Series(np.nan, index=frame.index, dtype=float)\n        for _, block in frame.groupby("instrument", sort=False):\n            rolling_corr.loc[block.index] = (\n                block["stock_return"]\n                .rolling(window, min_periods=min_periods)\n                .corr(block[f"market_lag_{lag}"])\n                .to_numpy()\n            )\n        squared_correlations.append(rolling_corr.pow(2))\n    total = sum(squared_correlations)\n    delay = sum(squared_correlations[1:]) / total.where(total > 0)\n    frame["factor_raw"] = -delay\n    frame["factor_raw"] = frame["factor_raw"].replace([np.inf, -np.inf], np.nan)\n    return frame\n\n\ndef build_pv_009_factor(\n    daily_bars: pd.DataFrame,\n    pool: pd.DataFrame,\n    *,\n    window: int = 252,\n    min_periods: int = 200,\n    lags: int = 4,\n) -> pd.DataFrame:\n    return build_ranked_factor(\n        compute_pv_009_daily(\n            daily_bars,\n            window=window,\n            min_periods=min_periods,\n            lags=lags,\n        ),\n        pool,\n        candidate_id="PV-009",\n    )\n', 'bigalpha2026.candidates.pv.pv_010': '"""PV-010: rolling market coskewness."""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom ._common import build_ranked_factor, prepare_daily\n\n\ndef compute_pv_010_daily(\n    daily_bars: pd.DataFrame,\n    *,\n    window: int = 252,\n    min_periods: int = 200,\n) -> pd.DataFrame:\n    frame = prepare_daily(\n        daily_bars,\n        ("date", "instrument", "close", "pre_close"),\n    )\n    frame["ri"] = (\n        frame["close"] / frame["pre_close"].where(frame["pre_close"] > 0) - 1.0\n    )\n    market = frame.groupby("date", sort=True)["ri"].mean().rename("rm")\n    frame = frame.merge(market, on="date", how="left", validate="many_to_one")\n    frame["ri_rm"] = frame["ri"] * frame["rm"]\n    frame["ri_rm2"] = frame["ri"] * frame["rm"].pow(2)\n    frame["rm2"] = frame["rm"].pow(2)\n\n    grouped = frame.groupby("instrument", sort=False)\n\n    def rolling_mean(column: str) -> pd.Series:\n        return (\n            grouped[column]\n            .rolling(window, min_periods=min_periods)\n            .mean()\n            .reset_index(level=0, drop=True)\n            .sort_index()\n        )\n\n    mean_ri = rolling_mean("ri")\n    mean_rm = rolling_mean("rm")\n    mean_ri_rm = rolling_mean("ri_rm")\n    mean_ri_rm2 = rolling_mean("ri_rm2")\n    mean_rm2 = rolling_mean("rm2")\n    var_ri = (\n        grouped["ri"]\n        .rolling(window, min_periods=min_periods)\n        .var()\n        .reset_index(level=0, drop=True)\n        .sort_index()\n    )\n    var_rm = mean_rm2 - mean_rm.pow(2)\n    numerator = (\n        mean_ri_rm2\n        - 2.0 * mean_rm * mean_ri_rm\n        - mean_ri * mean_rm2\n        + 2.0 * mean_ri * mean_rm.pow(2)\n    )\n    denominator = np.sqrt(var_ri.clip(lower=0)) * var_rm.clip(lower=0)\n    frame["factor_raw"] = -(numerator / denominator.where(denominator > 0))\n    frame["factor_raw"] = frame["factor_raw"].replace([np.inf, -np.inf], np.nan)\n    return frame\n\n\ndef build_pv_010_factor(\n    daily_bars: pd.DataFrame,\n    pool: pd.DataFrame,\n    *,\n    window: int = 252,\n    min_periods: int = 200,\n) -> pd.DataFrame:\n    return build_ranked_factor(\n        compute_pv_010_daily(\n            daily_bars,\n            window=window,\n            min_periods=min_periods,\n        ),\n        pool,\n        candidate_id="PV-010",\n    )\n', 'bigalpha2026.candidates.pv.pv_011': '"""PV-011: intermediate momentum from t-12 to t-6 months."""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom ._common import build_ranked_factor, prepare_daily\n\n\ndef compute_pv_011_daily(\n    daily_bars: pd.DataFrame,\n    *,\n    old_lag: int = 252,\n    recent_lag: int = 126,\n) -> pd.DataFrame:\n    frame = prepare_daily(daily_bars, ("date", "instrument", "close"))\n    grouped = frame.groupby("instrument", sort=False)["close"]\n    old_price = grouped.shift(old_lag)\n    recent_price = grouped.shift(recent_lag)\n    frame["factor_raw"] = recent_price / old_price.where(old_price > 0) - 1.0\n    frame["factor_raw"] = frame["factor_raw"].replace([np.inf, -np.inf], np.nan)\n    return frame\n\n\ndef build_pv_011_factor(\n    daily_bars: pd.DataFrame,\n    pool: pd.DataFrame,\n    *,\n    old_lag: int = 252,\n    recent_lag: int = 126,\n) -> pd.DataFrame:\n    return build_ranked_factor(\n        compute_pv_011_daily(\n            daily_bars,\n            old_lag=old_lag,\n            recent_lag=recent_lag,\n        ),\n        pool,\n        candidate_id="PV-011",\n    )\n', 'bigalpha2026.candidates.pv.pv_013': '"""PV-013: twelve-to-one-month momentum."""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom ._common import build_ranked_factor, prepare_daily\n\n\ndef compute_pv_013_daily(\n    daily_bars: pd.DataFrame,\n    *,\n    old_lag: int = 252,\n    recent_lag: int = 21,\n) -> pd.DataFrame:\n    frame = prepare_daily(daily_bars, ("date", "instrument", "close"))\n    grouped = frame.groupby("instrument", sort=False)["close"]\n    old_price = grouped.shift(old_lag)\n    recent_price = grouped.shift(recent_lag)\n    frame["factor_raw"] = recent_price / old_price.where(old_price > 0) - 1.0\n    frame["factor_raw"] = frame["factor_raw"].replace([np.inf, -np.inf], np.nan)\n    return frame\n\n\ndef build_pv_013_factor(\n    daily_bars: pd.DataFrame,\n    pool: pd.DataFrame,\n) -> pd.DataFrame:\n    return build_ranked_factor(\n        compute_pv_013_daily(daily_bars),\n        pool,\n        candidate_id="PV-013",\n    )\n', 'bigalpha2026.candidates.pv.pv_014': '"""PV-014: negative one-month CAPM residual volatility."""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom ._common import build_ranked_factor, prepare_daily\n\n\ndef compute_pv_014_daily(\n    daily_bars: pd.DataFrame,\n    *,\n    window: int = 21,\n    min_periods: int = 15,\n) -> pd.DataFrame:\n    frame = prepare_daily(\n        daily_bars,\n        ("date", "instrument", "close", "pre_close"),\n    )\n    frame["ri"] = (\n        frame["close"] / frame["pre_close"].where(frame["pre_close"] > 0) - 1.0\n    )\n    market = frame.groupby("date", sort=True)["ri"].mean().rename("rm")\n    frame = frame.merge(market, on="date", how="left", validate="many_to_one")\n    frame["ri2"] = frame["ri"].pow(2)\n    frame["rm2"] = frame["rm"].pow(2)\n    frame["ri_rm"] = frame["ri"] * frame["rm"]\n    grouped = frame.groupby("instrument", sort=False)\n\n    def rolling_mean(column: str) -> pd.Series:\n        return (\n            grouped[column]\n            .rolling(window, min_periods=min_periods)\n            .mean()\n            .reset_index(level=0, drop=True)\n            .sort_index()\n        )\n\n    mean_ri = rolling_mean("ri")\n    mean_rm = rolling_mean("rm")\n    var_ri = rolling_mean("ri2") - mean_ri.pow(2)\n    var_rm = rolling_mean("rm2") - mean_rm.pow(2)\n    cov = rolling_mean("ri_rm") - mean_ri * mean_rm\n    residual_variance = var_ri - cov.pow(2) / var_rm.where(var_rm > 0)\n    frame["factor_raw"] = -np.sqrt(residual_variance.clip(lower=0))\n    frame["factor_raw"] = frame["factor_raw"].replace([np.inf, -np.inf], np.nan)\n    return frame\n\n\ndef build_pv_014_factor(\n    daily_bars: pd.DataFrame,\n    pool: pd.DataFrame,\n) -> pd.DataFrame:\n    return build_ranked_factor(\n        compute_pv_014_daily(daily_bars),\n        pool,\n        candidate_id="PV-014",\n    )\n', 'bigalpha2026.candidates.pv.pv_015': '"""PV-015: negative 36-month volume variability."""\n\nfrom __future__ import annotations\n\nimport pandas as pd\n\nfrom ._monthly import build_monthly_state_factor, monthly_values\n\n\ndef compute_pv_015_monthly(\n    daily_bars: pd.DataFrame,\n    *,\n    window: int = 36,\n    min_periods: int = 24,\n) -> pd.DataFrame:\n    monthly = monthly_values(\n        daily_bars,\n        value_column="volume",\n        aggregation="sum",\n    )\n    monthly["factor_raw"] = -(\n        monthly.groupby("instrument", sort=False)["value"]\n        .rolling(window, min_periods=min_periods)\n        .std()\n        .reset_index(level=0, drop=True)\n        .sort_index()\n    )\n    return monthly\n\n\ndef build_pv_015_factor(\n    daily_bars: pd.DataFrame,\n    pool: pd.DataFrame,\n) -> pd.DataFrame:\n    return build_monthly_state_factor(\n        compute_pv_015_monthly(daily_bars),\n        pool,\n        candidate_id="PV-015",\n    )\n', 'bigalpha2026.candidates.pv.pv_016': '"""PV-016: negative 36-month turnover variability."""\n\nfrom __future__ import annotations\n\nimport pandas as pd\n\nfrom ._monthly import build_monthly_state_factor, monthly_values\n\n\ndef compute_pv_016_monthly(\n    factorlib: pd.DataFrame,\n    *,\n    window: int = 36,\n    min_periods: int = 24,\n) -> pd.DataFrame:\n    monthly = monthly_values(\n        factorlib,\n        value_column="turn",\n        aggregation="sum",\n    )\n    monthly["factor_raw"] = -(\n        monthly.groupby("instrument", sort=False)["value"]\n        .rolling(window, min_periods=min_periods)\n        .std()\n        .reset_index(level=0, drop=True)\n        .sort_index()\n    )\n    return monthly\n\n\ndef build_pv_016_factor(\n    factorlib: pd.DataFrame,\n    pool: pd.DataFrame,\n) -> pd.DataFrame:\n    return build_monthly_state_factor(\n        compute_pv_016_monthly(factorlib),\n        pool,\n        candidate_id="PV-016",\n    )\n', 'bigalpha2026.candidates.pv.pv_017': '"""PV-017: negative scaled 60-month volume trend."""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom ._monthly import build_monthly_state_factor, monthly_values\n\n\ndef _scaled_slope(values: np.ndarray) -> float:\n    valid = np.isfinite(values)\n    if valid.sum() < 2:\n        return np.nan\n    y = values[valid]\n    x = np.arange(len(values), dtype=float)[valid]\n    mean = float(np.mean(y))\n    if abs(mean) <= 1e-12:\n        return np.nan\n    slope = float(np.polyfit(x, y, 1)[0])\n    return -slope / abs(mean)\n\n\ndef compute_pv_017_monthly(\n    daily_bars: pd.DataFrame,\n    *,\n    window: int = 60,\n    min_periods: int = 30,\n) -> pd.DataFrame:\n    monthly = monthly_values(\n        daily_bars,\n        value_column="volume",\n        aggregation="sum",\n    )\n    monthly["factor_raw"] = (\n        monthly.groupby("instrument", sort=False)["value"]\n        .rolling(window, min_periods=min_periods)\n        .apply(_scaled_slope, raw=True)\n        .reset_index(level=0, drop=True)\n        .sort_index()\n    )\n    return monthly\n\n\ndef build_pv_017_factor(\n    daily_bars: pd.DataFrame,\n    pool: pd.DataFrame,\n) -> pd.DataFrame:\n    return build_monthly_state_factor(\n        compute_pv_017_monthly(daily_bars),\n        pool,\n        candidate_id="PV-017",\n    )\n', 'bigalpha2026.candidates.pv.pv_019': '"""PV-019: CAPM-residual momentum proxy."""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom ._common import build_ranked_factor, prepare_daily\n\n\ndef compute_pv_019_daily(\n    daily_bars: pd.DataFrame,\n    *,\n    beta_window: int = 252,\n    beta_min_periods: int = 200,\n    momentum_window: int = 231,\n    momentum_min_periods: int = 180,\n    skip_days: int = 21,\n) -> pd.DataFrame:\n    """Approximate FF3 residual momentum with an available CAPM residual."""\n\n    frame = prepare_daily(\n        daily_bars,\n        ("date", "instrument", "close", "pre_close"),\n    )\n    frame["ri"] = (\n        frame["close"] / frame["pre_close"].where(frame["pre_close"] > 0) - 1.0\n    )\n    market = frame.groupby("date", sort=True)["ri"].mean().rename("rm")\n    frame = frame.merge(market, on="date", how="left", validate="many_to_one")\n    frame["ri_rm"] = frame["ri"] * frame["rm"]\n    frame["rm2"] = frame["rm"].pow(2)\n    grouped = frame.groupby("instrument", sort=False)\n\n    def beta_mean(column: str) -> pd.Series:\n        return (\n            grouped[column]\n            .rolling(beta_window, min_periods=beta_min_periods)\n            .mean()\n            .reset_index(level=0, drop=True)\n            .sort_index()\n        )\n\n    mean_ri = beta_mean("ri")\n    mean_rm = beta_mean("rm")\n    cov = beta_mean("ri_rm") - mean_ri * mean_rm\n    var_rm = beta_mean("rm2") - mean_rm.pow(2)\n    beta = cov / var_rm.where(var_rm > 0)\n    frame["residual"] = frame["ri"] - (mean_ri + beta * (frame["rm"] - mean_rm))\n    frame["lagged_residual"] = grouped["residual"].shift(skip_days)\n    residual_grouped = frame.groupby("instrument", sort=False)["lagged_residual"]\n    rolling_mean = (\n        residual_grouped.rolling(\n            momentum_window,\n            min_periods=momentum_min_periods,\n        )\n        .mean()\n        .reset_index(level=0, drop=True)\n        .sort_index()\n    )\n    rolling_std = (\n        residual_grouped.rolling(\n            momentum_window,\n            min_periods=momentum_min_periods,\n        )\n        .std()\n        .reset_index(level=0, drop=True)\n        .sort_index()\n    )\n    frame["factor_raw"] = rolling_mean / rolling_std.where(rolling_std > 0)\n    frame["factor_raw"] = frame["factor_raw"].replace([np.inf, -np.inf], np.nan)\n    return frame\n\n\ndef build_pv_019_factor(\n    daily_bars: pd.DataFrame,\n    pool: pd.DataFrame,\n) -> pd.DataFrame:\n    return build_ranked_factor(\n        compute_pv_019_daily(daily_bars),\n        pool,\n        candidate_id="PV-019",\n    )\n', 'bigalpha2026.candidates.pv.pv_020': '"""PV-020: liquidity-conditioned short-term reversal."""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom ._common import build_ranked_factor, prepare_daily\n\n\ndef compute_pv_020_daily(\n    daily_bars: pd.DataFrame,\n    *,\n    history_window: int = 20,\n    min_periods: int = 10,\n) -> pd.DataFrame:\n    """Scale today\'s return shock by strictly lagged volatility and liquidity."""\n\n    if history_window < 2:\n        raise ValueError("history_window must be at least 2")\n    if min_periods < 2 or min_periods > history_window:\n        raise ValueError("min_periods must be between 2 and history_window")\n\n    frame = prepare_daily(\n        daily_bars,\n        ("date", "instrument", "close", "pre_close", "amount"),\n    )\n    frame["daily_return"] = frame["close"] / frame["pre_close"].where(frame["pre_close"] > 0) - 1.0\n    grouped = frame.groupby("instrument", sort=False)\n    prior_return = grouped["daily_return"].shift(1)\n    prior_amount = grouped["amount"].shift(1)\n    prior_volatility = (\n        prior_return.groupby(frame["instrument"], sort=False)\n        .rolling(history_window, min_periods=min_periods)\n        .std()\n        .reset_index(level=0, drop=True)\n        .sort_index()\n    )\n    prior_amount_median = (\n        prior_amount.groupby(frame["instrument"], sort=False)\n        .rolling(history_window, min_periods=min_periods)\n        .median()\n        .reset_index(level=0, drop=True)\n        .sort_index()\n    )\n\n    return_shock = (frame["daily_return"] / prior_volatility.where(prior_volatility > 1e-12)).clip(\n        -5.0, 5.0\n    )\n    liquidity_scarcity = (prior_amount_median / frame["amount"].where(frame["amount"] > 0)).clip(\n        0.25, 4.0\n    )\n    frame["factor_raw"] = (-return_shock * liquidity_scarcity).replace(\n        [np.inf, -np.inf],\n        np.nan,\n    )\n    return frame\n\n\ndef build_pv_020_factor(\n    daily_bars: pd.DataFrame,\n    pool: pd.DataFrame,\n) -> pd.DataFrame:\n    return build_ranked_factor(\n        compute_pv_020_daily(daily_bars),\n        pool,\n        candidate_id="PV-020",\n    )\n', 'bigalpha2026.candidates.pv.pv_021': '"""PV-021: dynamic volume-return regime."""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom ._common import build_ranked_factor, prepare_daily\n\n\ndef _lagged_rolling_mean(\n    frame: pd.DataFrame,\n    values: pd.Series,\n    *,\n    window: int,\n    min_periods: int,\n) -> pd.Series:\n    lagged = values.groupby(frame["instrument"], sort=False).shift(1)\n    return (\n        lagged.groupby(frame["instrument"], sort=False)\n        .rolling(window, min_periods=min_periods)\n        .mean()\n        .reset_index(level=0, drop=True)\n        .sort_index()\n    )\n\n\ndef compute_pv_021_daily(\n    daily_bars: pd.DataFrame,\n    *,\n    amount_window: int = 20,\n    amount_min_periods: int = 10,\n    regression_window: int = 120,\n    regression_min_periods: int = 60,\n) -> pd.DataFrame:\n    """Estimate the lagged volume-return interaction without future observations."""\n\n    if amount_window < 2 or regression_window < 3:\n        raise ValueError("rolling windows are too short")\n    if amount_min_periods < 2 or amount_min_periods > amount_window:\n        raise ValueError("invalid amount_min_periods")\n    if regression_min_periods < 3 or regression_min_periods > regression_window:\n        raise ValueError("invalid regression_min_periods")\n\n    frame = prepare_daily(\n        daily_bars,\n        ("date", "instrument", "close", "pre_close", "amount"),\n    )\n    frame["daily_return"] = (\n        frame["close"] / frame["pre_close"].where(frame["pre_close"] > 0) - 1.0\n    )\n    grouped = frame.groupby("instrument", sort=False)\n    prior_amount = grouped["amount"].shift(1)\n    prior_amount_median = (\n        prior_amount.groupby(frame["instrument"], sort=False)\n        .rolling(amount_window, min_periods=amount_min_periods)\n        .median()\n        .reset_index(level=0, drop=True)\n        .sort_index()\n    )\n    frame["abnormal_volume"] = np.log(\n        frame["amount"].where(frame["amount"] > 0)\n        / prior_amount_median.where(prior_amount_median > 0)\n    ).clip(-3.0, 3.0)\n    frame["lagged_return"] = grouped["daily_return"].shift(1)\n    frame["interaction"] = frame["lagged_return"] * frame["abnormal_volume"]\n\n    y = frame["daily_return"]\n    x1 = frame["lagged_return"]\n    x2 = frame["interaction"]\n    means = {\n        "y": _lagged_rolling_mean(\n            frame,\n            y,\n            window=regression_window,\n            min_periods=regression_min_periods,\n        ),\n        "x1": _lagged_rolling_mean(\n            frame,\n            x1,\n            window=regression_window,\n            min_periods=regression_min_periods,\n        ),\n        "x2": _lagged_rolling_mean(\n            frame,\n            x2,\n            window=regression_window,\n            min_periods=regression_min_periods,\n        ),\n        "yx1": _lagged_rolling_mean(\n            frame,\n            y * x1,\n            window=regression_window,\n            min_periods=regression_min_periods,\n        ),\n        "yx2": _lagged_rolling_mean(\n            frame,\n            y * x2,\n            window=regression_window,\n            min_periods=regression_min_periods,\n        ),\n        "x1x2": _lagged_rolling_mean(\n            frame,\n            x1 * x2,\n            window=regression_window,\n            min_periods=regression_min_periods,\n        ),\n        "x1_sq": _lagged_rolling_mean(\n            frame,\n            x1.pow(2),\n            window=regression_window,\n            min_periods=regression_min_periods,\n        ),\n        "x2_sq": _lagged_rolling_mean(\n            frame,\n            x2.pow(2),\n            window=regression_window,\n            min_periods=regression_min_periods,\n        ),\n    }\n    cov_y_x1 = means["yx1"] - means["y"] * means["x1"]\n    cov_y_x2 = means["yx2"] - means["y"] * means["x2"]\n    cov_x1_x2 = means["x1x2"] - means["x1"] * means["x2"]\n    var_x1 = means["x1_sq"] - means["x1"].pow(2)\n    var_x2 = means["x2_sq"] - means["x2"].pow(2)\n    determinant = var_x1 * var_x2 - cov_x1_x2.pow(2)\n    interaction_coefficient = (\n        cov_y_x2 * var_x1 - cov_y_x1 * cov_x1_x2\n    ) / determinant.where(determinant > 1e-16)\n    frame["interaction_coefficient"] = interaction_coefficient.clip(-10.0, 10.0)\n    frame["factor_raw"] = (\n        frame["interaction_coefficient"]\n        * frame["daily_return"]\n        * frame["abnormal_volume"]\n    ).replace([np.inf, -np.inf], np.nan)\n    return frame\n\n\ndef build_pv_021_factor(\n    daily_bars: pd.DataFrame,\n    pool: pd.DataFrame,\n) -> pd.DataFrame:\n    return build_ranked_factor(\n        compute_pv_021_daily(daily_bars),\n        pool,\n        candidate_id="PV-021",\n    )\n', 'bigalpha2026.candidates.pv.pv_022': '"""PV-022: frog-in-the-pan momentum with continuous information."""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom ._common import build_ranked_factor, prepare_daily\n\n\ndef _rolling_share(\n    frame: pd.DataFrame,\n    column: str,\n    *,\n    shift_days: int,\n    window: int,\n    min_periods: int,\n) -> pd.Series:\n    return frame.groupby("instrument", sort=False)[column].transform(\n        lambda values: values.shift(shift_days).rolling(\n            window,\n            min_periods=min_periods,\n        ).mean()\n    )\n\n\ndef compute_pv_022_daily(\n    daily_bars: pd.DataFrame,\n    *,\n    old_lag: int = 252,\n    recent_lag: int = 21,\n    min_periods: int = 187,\n) -> pd.DataFrame:\n    """Scale twelve-to-one-month momentum by information continuity."""\n\n    if not 0 < recent_lag < old_lag:\n        raise ValueError("lags must satisfy 0 < recent_lag < old_lag")\n    formation_days = old_lag - recent_lag\n    if not 2 <= min_periods <= formation_days:\n        raise ValueError("min_periods must fit inside the formation window")\n\n    frame = prepare_daily(\n        daily_bars,\n        ("date", "instrument", "close", "pre_close"),\n    )\n    valid_price = frame["close"].gt(0) & frame["pre_close"].gt(0)\n    frame["daily_return"] = (\n        frame["close"] / frame["pre_close"] - 1.0\n    ).where(valid_price)\n    grouped_close = frame.groupby("instrument", sort=False)["close"]\n    old_price = grouped_close.shift(old_lag)\n    recent_price = grouped_close.shift(recent_lag)\n    frame["formation_return"] = (\n        recent_price / old_price.where(old_price > 0) - 1.0\n    )\n\n    observed = frame["daily_return"].notna()\n    frame["_positive"] = frame["daily_return"].gt(0).astype(float).where(observed)\n    frame["_negative"] = frame["daily_return"].lt(0).astype(float).where(observed)\n    positive_share = _rolling_share(\n        frame,\n        "_positive",\n        shift_days=recent_lag,\n        window=formation_days,\n        min_periods=min_periods,\n    )\n    negative_share = _rolling_share(\n        frame,\n        "_negative",\n        shift_days=recent_lag,\n        window=formation_days,\n        min_periods=min_periods,\n    )\n    frame["information_discreteness"] = np.sign(\n        frame["formation_return"]\n    ) * (negative_share - positive_share)\n    continuity_weight = (\n        1.0 - frame["information_discreteness"].clip(-1.0, 1.0)\n    ) / 2.0\n    frame["factor_raw"] = frame["formation_return"] * continuity_weight\n    frame["factor_raw"] = frame["factor_raw"].replace(\n        [np.inf, -np.inf],\n        np.nan,\n    )\n    return frame[\n        [\n            "date",\n            "instrument",\n            "formation_return",\n            "information_discreteness",\n            "factor_raw",\n        ]\n    ]\n\n\ndef build_pv_022_factor(\n    daily_bars: pd.DataFrame,\n    pool: pd.DataFrame,\n) -> pd.DataFrame:\n    return build_ranked_factor(\n        compute_pv_022_daily(daily_bars),\n        pool,\n        candidate_id="PV-022",\n    )\n', 'bigalpha2026.candidates.pv.pv_023': '"""PV-023: abnormal positive-overnight/daytime-reversal intensity."""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom ._common import build_ranked_factor, prepare_daily\n\n\ndef compute_pv_023_daily(\n    daily_bars: pd.DataFrame,\n    *,\n    window: int = 20,\n    min_periods: int = 15,\n) -> pd.DataFrame:\n    """Measure excess co-occurrence of positive nights and negative days."""\n\n    if window < 2:\n        raise ValueError("window must be at least 2")\n    if not 2 <= min_periods <= window:\n        raise ValueError("min_periods must be between 2 and window")\n\n    frame = prepare_daily(\n        daily_bars,\n        ("date", "instrument", "open", "close", "pre_close"),\n    )\n    valid_overnight = frame["open"].gt(0) & frame["pre_close"].gt(0)\n    valid_intraday = frame["close"].gt(0) & frame["open"].gt(0)\n    frame["overnight_return"] = (\n        frame["open"] / frame["pre_close"] - 1.0\n    ).where(valid_overnight)\n    frame["intraday_return"] = (\n        frame["close"] / frame["open"] - 1.0\n    ).where(valid_intraday)\n    observed = frame["overnight_return"].notna() & frame["intraday_return"].notna()\n    frame["_positive_overnight"] = (\n        frame["overnight_return"].gt(0).astype(float).where(observed)\n    )\n    frame["_negative_intraday"] = (\n        frame["intraday_return"].lt(0).astype(float).where(observed)\n    )\n    frame["_high_open_reversal"] = (\n        (\n            frame["overnight_return"].gt(0)\n            & frame["intraday_return"].lt(0)\n        )\n        .astype(float)\n        .where(observed)\n    )\n    group = frame.groupby("instrument", sort=False)\n    rolling: dict[str, pd.Series] = {}\n    for column in (\n        "_positive_overnight",\n        "_negative_intraday",\n        "_high_open_reversal",\n    ):\n        rolling[column] = group[column].transform(\n            lambda values: values.rolling(\n                window,\n                min_periods=min_periods,\n            ).mean()\n        )\n    expected_frequency = (\n        rolling["_positive_overnight"] * rolling["_negative_intraday"]\n    )\n    frame["factor_raw"] = rolling["_high_open_reversal"] - expected_frequency\n    frame["factor_raw"] = frame["factor_raw"].replace(\n        [np.inf, -np.inf],\n        np.nan,\n    )\n    return frame[\n        [\n            "date",\n            "instrument",\n            "overnight_return",\n            "intraday_return",\n            "factor_raw",\n        ]\n    ]\n\n\ndef build_pv_023_factor(\n    daily_bars: pd.DataFrame,\n    pool: pd.DataFrame,\n) -> pd.DataFrame:\n    return build_ranked_factor(\n        compute_pv_023_daily(daily_bars),\n        pool,\n        candidate_id="PV-023",\n    )\n', 'bigalpha2026.candidates.hf.hf_001': '"""HF-001: intraday shock absorption and recovery.\n\nReal data queries and empirical evaluation belong in BigQuant AIStudio. This\nmodule only defines deterministic data-frame transformations.\n"""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Iterable\n\nimport numpy as np\nimport pandas as pd\n\nBAR_COLUMNS = ("date", "instrument", "close", "amount", "volume", "deal_number")\nPOOL_COLUMNS = ("date", "instrument")\nOUTPUT_COLUMNS = ("date", "instrument", "factor")\n\n\ndef _require_columns(frame: pd.DataFrame, columns: Iterable[str], name: str) -> None:\n    missing = sorted(set(columns).difference(frame.columns))\n    if missing:\n        raise ValueError(f"{name} is missing required columns: {missing}")\n\n\ndef compute_hf_001_daily(\n    minute_bars: pd.DataFrame,\n    *,\n    recovery_minutes: int = 5,\n    shock_quantile: float = 0.90,\n    min_shocks: int = 3,\n) -> pd.DataFrame:\n    """Measure whether high-activity price shocks reverse within the session."""\n\n    _require_columns(minute_bars, BAR_COLUMNS, "minute_bars")\n    if recovery_minutes < 1:\n        raise ValueError("recovery_minutes must be positive")\n    if not 0.5 < shock_quantile < 1.0:\n        raise ValueError("shock_quantile must be between 0.5 and 1")\n    if min_shocks < 1:\n        raise ValueError("min_shocks must be positive")\n\n    frame = minute_bars.loc[:, BAR_COLUMNS].copy()\n    frame["timestamp"] = pd.to_datetime(frame["date"], errors="coerce")\n    frame["date"] = frame["timestamp"].dt.normalize()\n    frame["instrument"] = frame["instrument"].astype(str)\n    for column in ("close", "amount", "volume", "deal_number"):\n        frame[column] = pd.to_numeric(frame[column], errors="coerce")\n    frame = frame.dropna(subset=["timestamp", "instrument"]).sort_values(\n        ["instrument", "timestamp"]\n    )\n    frame["session"] = np.where(frame["timestamp"].dt.hour < 12, "morning", "afternoon")\n    session_keys = ["date", "instrument", "session"]\n    session_group = frame.groupby(session_keys, sort=False)\n    previous_close = session_group["close"].shift(1)\n    future_close = session_group["close"].shift(-recovery_minutes)\n    frame["minute_return"] = frame["close"] / previous_close - 1.0\n    frame["future_return"] = future_close / frame["close"] - 1.0\n\n    day_group = frame.groupby(["date", "instrument"], sort=False)\n    frame["shock_cutoff"] = day_group["minute_return"].transform(\n        lambda values: values.abs().quantile(shock_quantile)\n    )\n    activity = (\n        np.log1p(frame["amount"].clip(lower=0))\n        + np.log1p(frame["volume"].clip(lower=0))\n        + np.log1p(frame["deal_number"].clip(lower=0))\n    ) / 3.0\n    frame["activity"] = activity\n    frame["activity_median"] = day_group["activity"].transform("median")\n    shock = (\n        frame["minute_return"].abs().ge(frame["shock_cutoff"])\n        & frame["activity"].ge(frame["activity_median"])\n        & frame["future_return"].notna()\n        & frame["minute_return"].ne(0)\n    )\n    selected = frame.loc[shock].copy()\n    selected["recovery_ratio"] = (\n        -np.sign(selected["minute_return"])\n        * selected["future_return"]\n        / selected["minute_return"].abs()\n    ).clip(-2.0, 2.0)\n\n    daily = (\n        selected.groupby(["date", "instrument"], sort=False)\n        .agg(\n            factor_raw=("recovery_ratio", "median"),\n            shock_count=("recovery_ratio", "size"),\n            mean_shock=("minute_return", lambda values: values.abs().mean()),\n        )\n        .reset_index()\n    )\n    daily.loc[daily["shock_count"] < min_shocks, "factor_raw"] = np.nan\n    daily["factor_raw"] = daily["factor_raw"].replace([np.inf, -np.inf], np.nan)\n    return daily.sort_values(["instrument", "date"]).reset_index(drop=True)\n\n\ndef build_hf_001_factor(\n    minute_bars: pd.DataFrame,\n    pool: pd.DataFrame,\n    *,\n    start_date: object | None = None,\n    end_date: object | None = None,\n    recovery_minutes: int = 5,\n    shock_quantile: float = 0.90,\n    min_shocks: int = 3,\n) -> pd.DataFrame:\n    """Return the exact ``date, instrument, factor`` research interface."""\n\n    _require_columns(pool, POOL_COLUMNS, "pool")\n    daily = compute_hf_001_daily(\n        minute_bars,\n        recovery_minutes=recovery_minutes,\n        shock_quantile=shock_quantile,\n        min_shocks=min_shocks,\n    )\n    panel = pool.loc[:, POOL_COLUMNS].copy()\n    panel["date"] = pd.to_datetime(panel["date"], errors="coerce").dt.normalize().astype("datetime64[ns]")\n    panel["instrument"] = panel["instrument"].astype(str)\n    panel = panel.dropna(subset=["date", "instrument"])\n    if start_date is not None:\n        panel = panel.loc[panel["date"] >= pd.Timestamp(start_date).normalize()]\n    if end_date is not None:\n        panel = panel.loc[panel["date"] <= pd.Timestamp(end_date).normalize()]\n\n    result = panel.merge(\n        daily[["date", "instrument", "factor_raw"]],\n        on=["date", "instrument"],\n        how="left",\n        validate="one_to_one",\n    )\n    daily_median = result.groupby("date", sort=False)["factor_raw"].transform("median")\n    result["factor_raw"] = result["factor_raw"].fillna(daily_median).fillna(0.0)\n    result["factor"] = (\n        result.groupby("date", sort=False)["factor_raw"]\n        .rank(pct=True, method="average")\n        .sub(0.5)\n        .mul(2.0)\n    )\n    result["factor"] = pd.to_numeric(result["factor"], errors="coerce").replace(\n        [np.inf, -np.inf],\n        np.nan,\n    )\n    if result["factor"].isna().any():\n        raise ValueError("HF-001 produced non-finite factor values")\n    return (\n        result.loc[:, OUTPUT_COLUMNS]\n        .drop_duplicates(["date", "instrument"], keep="last")\n        .sort_values(["date", "instrument"])\n        .reset_index(drop=True)\n    )\n\n\ndef build_hf_001_factor_from_daily(\n    daily_features: pd.DataFrame,\n    pool: pd.DataFrame,\n) -> pd.DataFrame:\n    """Build HF-001 from the frozen AIStudio daily component panel."""\n\n    required = (\n        "date",\n        "instrument",\n        "shock_q90_active_count",\n        "shock_q90_recovery_5m_median",\n    )\n    _require_columns(daily_features, required, "daily_features")\n    _require_columns(pool, POOL_COLUMNS, "pool")\n    daily = daily_features.loc[:, required].copy()\n    daily["date"] = pd.to_datetime(daily["date"], errors="coerce").dt.normalize()\n    daily["instrument"] = daily["instrument"].astype(str)\n    daily["factor_raw"] = pd.to_numeric(\n        daily["shock_q90_recovery_5m_median"], errors="coerce"\n    )\n    shock_count = pd.to_numeric(daily["shock_q90_active_count"], errors="coerce")\n    daily.loc[shock_count < 3, "factor_raw"] = np.nan\n\n    panel = pool.loc[:, POOL_COLUMNS].copy()\n    panel["date"] = pd.to_datetime(panel["date"], errors="coerce").dt.normalize().astype("datetime64[ns]")\n    panel["instrument"] = panel["instrument"].astype(str)\n    result = panel.merge(\n        daily[["date", "instrument", "factor_raw"]],\n        on=["date", "instrument"],\n        how="left",\n        validate="one_to_one",\n    )\n    median = result.groupby("date", sort=False)["factor_raw"].transform("median")\n    result["factor_raw"] = result["factor_raw"].fillna(median).fillna(0.0)\n    result["factor"] = (\n        result.groupby("date", sort=False)["factor_raw"]\n        .rank(pct=True, method="average")\n        .sub(0.5)\n        .mul(2.0)\n    )\n    if not np.isfinite(result["factor"]).all():\n        raise ValueError("HF-001 produced non-finite factor values")\n    return result.loc[:, OUTPUT_COLUMNS].sort_values(\n        ["date", "instrument"]\n    ).reset_index(drop=True)\n', 'bigalpha2026.candidates.hf.hf_003': '"""HF-003: relative signed intraday jump variation."""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Iterable\n\nimport numpy as np\nimport pandas as pd\n\nPOOL_COLUMNS = ("date", "instrument")\nOUTPUT_COLUMNS = ("date", "instrument", "factor")\nREALIZED_VOLATILITY = "realized_volatility"\nDOWNSIDE_REALIZED_VOLATILITY = "downside_realized_volatility"\n\n\ndef _require_columns(frame: pd.DataFrame, columns: Iterable[str], name: str) -> None:\n    missing = sorted(set(columns).difference(frame.columns))\n    if missing:\n        raise ValueError(f"{name} is missing required columns: {missing}")\n\n\ndef compute_hf_003_daily(\n    daily_features: pd.DataFrame,\n    *,\n    lookback_days: int = 5,\n    min_periods: int = 3,\n) -> pd.DataFrame:\n    """Compute the negative rolling mean of relative signed variation."""\n\n    required = (\n        *POOL_COLUMNS,\n        REALIZED_VOLATILITY,\n        DOWNSIDE_REALIZED_VOLATILITY,\n    )\n    _require_columns(daily_features, required, "daily_features")\n    if lookback_days < 1:\n        raise ValueError("lookback_days must be positive")\n    if not 1 <= min_periods <= lookback_days:\n        raise ValueError("min_periods must be between 1 and lookback_days")\n\n    daily = daily_features.loc[:, required].copy()\n    daily["date"] = pd.to_datetime(daily["date"], errors="coerce").dt.normalize()\n    daily["instrument"] = daily["instrument"].astype(str)\n    if daily.duplicated(list(POOL_COLUMNS)).any():\n        raise ValueError("daily_features contains duplicate date-instrument keys")\n    daily = daily.sort_values(["instrument", "date"]).reset_index(drop=True)\n\n    realized_variation = pd.to_numeric(\n        daily[REALIZED_VOLATILITY], errors="coerce"\n    ).pow(2)\n    downside_variation = pd.to_numeric(\n        daily[DOWNSIDE_REALIZED_VOLATILITY], errors="coerce"\n    ).pow(2)\n    valid = (\n        realized_variation.gt(0)\n        & downside_variation.ge(0)\n        & downside_variation.le(realized_variation * (1.0 + 1e-9))\n    )\n    downside_variation = downside_variation.clip(upper=realized_variation)\n    daily["relative_signed_variation"] = (\n        1.0 - 2.0 * downside_variation / realized_variation\n    ).where(valid)\n    daily["factor_raw"] = daily.groupby(\n        "instrument", sort=False\n    )["relative_signed_variation"].transform(\n        lambda values: -values.rolling(\n            lookback_days,\n            min_periods=min_periods,\n        ).mean()\n    )\n    daily["factor_raw"] = daily["factor_raw"].replace(\n        [np.inf, -np.inf],\n        np.nan,\n    )\n    return daily[\n        [\n            "date",\n            "instrument",\n            "relative_signed_variation",\n            "factor_raw",\n        ]\n    ]\n\n\ndef build_hf_003_factor_from_daily(\n    daily_features: pd.DataFrame,\n    pool: pd.DataFrame,\n    *,\n    lookback_days: int = 5,\n    min_periods: int = 3,\n) -> pd.DataFrame:\n    """Build HF-003 from the frozen AIStudio daily microstructure panel."""\n\n    _require_columns(pool, POOL_COLUMNS, "pool")\n    daily = compute_hf_003_daily(\n        daily_features,\n        lookback_days=lookback_days,\n        min_periods=min_periods,\n    )\n    panel = pool.loc[:, POOL_COLUMNS].copy()\n    panel["date"] = pd.to_datetime(panel["date"], errors="coerce").dt.normalize().astype("datetime64[ns]")\n    panel["instrument"] = panel["instrument"].astype(str)\n    panel = panel.dropna(subset=list(POOL_COLUMNS))\n    if panel.duplicated(list(POOL_COLUMNS)).any():\n        raise ValueError("pool contains duplicate date-instrument keys")\n\n    result = panel.merge(\n        daily[["date", "instrument", "factor_raw"]],\n        on=list(POOL_COLUMNS),\n        how="left",\n        validate="one_to_one",\n    )\n    median = result.groupby("date", sort=False)["factor_raw"].transform("median")\n    result["factor_raw"] = result["factor_raw"].fillna(median).fillna(0.0)\n    result["factor"] = (\n        result.groupby("date", sort=False)["factor_raw"]\n        .rank(pct=True, method="average")\n        .sub(0.5)\n        .mul(2.0)\n    )\n    if not np.isfinite(result["factor"]).all():\n        raise ValueError("HF-003 produced non-finite factor values")\n    return result.loc[:, OUTPUT_COLUMNS].sort_values(\n        ["date", "instrument"]\n    ).reset_index(drop=True)\n', 'bigalpha2026.candidates.hf.hf_004': '"""HF-004: residual closing signed-volume pressure."""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Iterable\n\nimport numpy as np\nimport pandas as pd\n\nPOOL_COLUMNS = ("date", "instrument")\nOUTPUT_COLUMNS = ("date", "instrument", "factor")\nTAIL_RETURN = "tail_60_log_return"\nTAIL_SIGNED_VOLUME = "tail_60_signed_volume_bvc"\nTAIL_VOLUME = "tail_60_volume"\n\n\ndef _require_columns(frame: pd.DataFrame, columns: Iterable[str], name: str) -> None:\n    missing = sorted(set(columns).difference(frame.columns))\n    if missing:\n        raise ValueError(f"{name} is missing required columns: {missing}")\n\n\ndef _lagged_rolling_mean(\n    values: pd.Series,\n    *,\n    window: int,\n    min_periods: int,\n) -> pd.Series:\n    return values.shift(1).rolling(window, min_periods=min_periods).mean()\n\n\ndef compute_hf_004_daily(\n    daily_features: pd.DataFrame,\n    *,\n    regression_window_days: int = 60,\n    regression_min_periods: int = 30,\n) -> pd.DataFrame:\n    """Remove the price-explained component of closing signed-volume pressure."""\n\n    required = (*POOL_COLUMNS, TAIL_RETURN, TAIL_SIGNED_VOLUME, TAIL_VOLUME)\n    _require_columns(daily_features, required, "daily_features")\n    if regression_window_days < 2:\n        raise ValueError("regression_window_days must be at least 2")\n    if not 2 <= regression_min_periods <= regression_window_days:\n        raise ValueError(\n            "regression_min_periods must be between 2 and regression_window_days"\n        )\n\n    daily = daily_features.loc[:, required].copy()\n    daily["date"] = pd.to_datetime(daily["date"], errors="coerce").dt.normalize()\n    daily["instrument"] = daily["instrument"].astype(str)\n    if daily.duplicated(list(POOL_COLUMNS)).any():\n        raise ValueError("daily_features contains duplicate date-instrument keys")\n    daily = daily.sort_values(["instrument", "date"]).reset_index(drop=True)\n\n    tail_return = pd.to_numeric(daily[TAIL_RETURN], errors="coerce")\n    tail_signed_volume = pd.to_numeric(\n        daily[TAIL_SIGNED_VOLUME], errors="coerce"\n    )\n    tail_volume = pd.to_numeric(daily[TAIL_VOLUME], errors="coerce")\n    daily["tail_return"] = tail_return\n    daily["tail_order_imbalance"] = (\n        tail_signed_volume / tail_volume.where(tail_volume > 0)\n    ).clip(-1.0, 1.0)\n\n    paired = daily["tail_return"].notna() & daily["tail_order_imbalance"].notna()\n    daily["_x"] = daily["tail_return"].where(paired)\n    daily["_y"] = daily["tail_order_imbalance"].where(paired)\n    daily["_xx"] = daily["_x"].pow(2)\n    daily["_xy"] = daily["_x"] * daily["_y"]\n    group = daily.groupby("instrument", sort=False)\n    rolling: dict[str, pd.Series] = {}\n    for column in ("_x", "_y", "_xx", "_xy"):\n        rolling[column] = group[column].transform(\n            lambda values: _lagged_rolling_mean(\n                values,\n                window=regression_window_days,\n                min_periods=regression_min_periods,\n            )\n        )\n    covariance = rolling["_xy"] - rolling["_x"] * rolling["_y"]\n    variance = rolling["_xx"] - rolling["_x"].pow(2)\n    beta = covariance / variance.where(variance > 1e-12)\n    alpha = rolling["_y"] - beta * rolling["_x"]\n    daily["factor_raw"] = daily["_y"] - (\n        alpha + beta * daily["_x"]\n    )\n    daily["factor_raw"] = daily["factor_raw"].replace(\n        [np.inf, -np.inf],\n        np.nan,\n    )\n    return daily[\n        [\n            "date",\n            "instrument",\n            "tail_return",\n            "tail_order_imbalance",\n            "factor_raw",\n        ]\n    ]\n\n\ndef build_hf_004_factor_from_daily(\n    daily_features: pd.DataFrame,\n    pool: pd.DataFrame,\n    *,\n    regression_window_days: int = 60,\n    regression_min_periods: int = 30,\n) -> pd.DataFrame:\n    """Build HF-004 from the extended AIStudio daily microstructure panel."""\n\n    _require_columns(pool, POOL_COLUMNS, "pool")\n    daily = compute_hf_004_daily(\n        daily_features,\n        regression_window_days=regression_window_days,\n        regression_min_periods=regression_min_periods,\n    )\n    panel = pool.loc[:, POOL_COLUMNS].copy()\n    panel["date"] = pd.to_datetime(panel["date"], errors="coerce").dt.normalize().astype("datetime64[ns]")\n    panel["instrument"] = panel["instrument"].astype(str)\n    panel = panel.dropna(subset=list(POOL_COLUMNS))\n    if panel.duplicated(list(POOL_COLUMNS)).any():\n        raise ValueError("pool contains duplicate date-instrument keys")\n\n    result = panel.merge(\n        daily[["date", "instrument", "factor_raw"]],\n        on=list(POOL_COLUMNS),\n        how="left",\n        validate="one_to_one",\n    )\n    median = result.groupby("date", sort=False)["factor_raw"].transform("median")\n    result["factor_raw"] = result["factor_raw"].fillna(median).fillna(0.0)\n    result["factor"] = (\n        result.groupby("date", sort=False)["factor_raw"]\n        .rank(pct=True, method="average")\n        .sub(0.5)\n        .mul(2.0)\n    )\n    if not np.isfinite(result["factor"]).all():\n        raise ValueError("HF-004 produced non-finite factor values")\n    return result.loc[:, OUTPUT_COLUMNS].sort_values(\n        ["date", "instrument"]\n    ).reset_index(drop=True)\n', 'bigalpha2026.candidates.ob.ob_001': '"""OB-001: valid-depth order-book resilience.\n\nZero-price or zero-volume levels are treated as absent quotes. Real-data\nqueries and empirical evaluation must run in BigQuant AIStudio.\n"""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Iterable\n\nimport numpy as np\nimport pandas as pd\n\nLEVELS = range(1, 6)\nPRICE_COLUMNS = tuple(\n    column\n    for level in LEVELS\n    for column in (f"bid_price{level}", f"ask_price{level}")\n)\nVOLUME_COLUMNS = tuple(\n    column\n    for level in LEVELS\n    for column in (f"bid_volume{level}", f"ask_volume{level}")\n)\nBAR_COLUMNS = ("date", "instrument", *PRICE_COLUMNS, *VOLUME_COLUMNS)\nPOOL_COLUMNS = ("date", "instrument")\nOUTPUT_COLUMNS = ("date", "instrument", "factor")\nCOMPONENT_COLUMNS = (\n    "spread_close",\n    "depth_completeness",\n    "bid_imbalance",\n    "bid_recovery",\n)\n\n\ndef _require_columns(frame: pd.DataFrame, columns: Iterable[str], name: str) -> None:\n    missing = sorted(set(columns).difference(frame.columns))\n    if missing:\n        raise ValueError(f"{name} is missing required columns: {missing}")\n\n\ndef compute_ob_001_daily(\n    minute_bars: pd.DataFrame,\n    *,\n    tail_minutes: int = 60,\n    recovery_minutes: int = 5,\n    shock_quantile: float = 0.10,\n    min_valid_tail_minutes: int = 30,\n) -> pd.DataFrame:\n    """Compute quote-state and bid-depth recovery components per stock-day."""\n\n    _require_columns(minute_bars, BAR_COLUMNS, "minute_bars")\n    if tail_minutes < 1 or recovery_minutes < 1:\n        raise ValueError("tail_minutes and recovery_minutes must be positive")\n    if not 0.0 < shock_quantile < 0.5:\n        raise ValueError("shock_quantile must be between 0 and 0.5")\n\n    frame = minute_bars.loc[:, BAR_COLUMNS].copy()\n    frame["timestamp"] = pd.to_datetime(frame["date"], errors="coerce")\n    frame["date"] = frame["timestamp"].dt.normalize()\n    frame["instrument"] = frame["instrument"].astype(str)\n    for column in (*PRICE_COLUMNS, *VOLUME_COLUMNS):\n        frame[column] = pd.to_numeric(frame[column], errors="coerce")\n    frame = frame.dropna(subset=["timestamp", "instrument"]).sort_values(\n        ["instrument", "timestamp"]\n    )\n\n    bid_depth = pd.Series(0.0, index=frame.index)\n    ask_depth = pd.Series(0.0, index=frame.index)\n    valid_bid_count = pd.Series(0, index=frame.index)\n    valid_ask_count = pd.Series(0, index=frame.index)\n    for level in LEVELS:\n        valid_bid = (frame[f"bid_price{level}"] > 0) & (frame[f"bid_volume{level}"] > 0)\n        valid_ask = (frame[f"ask_price{level}"] > 0) & (frame[f"ask_volume{level}"] > 0)\n        bid_depth = bid_depth + frame[f"bid_volume{level}"].where(valid_bid, 0.0)\n        ask_depth = ask_depth + frame[f"ask_volume{level}"].where(valid_ask, 0.0)\n        valid_bid_count = valid_bid_count + valid_bid.astype(int)\n        valid_ask_count = valid_ask_count + valid_ask.astype(int)\n\n    frame["bid_depth"] = bid_depth\n    frame["ask_depth"] = ask_depth\n    frame["valid_bid_count"] = valid_bid_count\n    frame["valid_ask_count"] = valid_ask_count\n    best_valid = (\n        (frame["bid_price1"] > 0)\n        & (frame["ask_price1"] > 0)\n        & (frame["bid_volume1"] > 0)\n        & (frame["ask_volume1"] > 0)\n        & (frame["ask_price1"] >= frame["bid_price1"])\n    )\n    frame["mid_price"] = ((frame["ask_price1"] + frame["bid_price1"]) / 2.0).where(\n        best_valid\n    )\n    frame["relative_spread"] = (\n        (frame["ask_price1"] - frame["bid_price1"]) / frame["mid_price"]\n    ).where(best_valid)\n    frame["depth_completeness"] = (\n        np.minimum(frame["valid_bid_count"], frame["valid_ask_count"]) / 5.0\n    )\n    frame["bid_imbalance"] = (\n        (frame["bid_depth"] - frame["ask_depth"])\n        / (frame["bid_depth"] + frame["ask_depth"]).replace(0, np.nan)\n    )\n\n    frame["session"] = np.where(frame["timestamp"].dt.hour < 12, "morning", "afternoon")\n    session_group = frame.groupby(["date", "instrument", "session"], sort=False)\n    previous_mid = session_group["mid_price"].shift(1)\n    future_bid_depth = session_group["bid_depth"].shift(-recovery_minutes)\n    frame["mid_return"] = frame["mid_price"] / previous_mid - 1.0\n    frame["future_bid_recovery"] = (\n        future_bid_depth / frame["bid_depth"].replace(0, np.nan) - 1.0\n    ).clip(-2.0, 2.0)\n\n    day_group = frame.groupby(["date", "instrument"], sort=False)\n    frame["negative_shock_cutoff"] = day_group["mid_return"].transform(\n        lambda values: values.quantile(shock_quantile)\n    )\n    negative_shock = (\n        frame["mid_return"].lt(0)\n        & frame["mid_return"].le(frame["negative_shock_cutoff"])\n        & frame["future_bid_recovery"].notna()\n    )\n    recovery = (\n        frame.loc[negative_shock]\n        .groupby(["date", "instrument"], sort=False)["future_bid_recovery"]\n        .median()\n        .rename("bid_recovery")\n        .reset_index()\n    )\n\n    frame["reverse_minute"] = day_group.cumcount(ascending=False) + 1\n    tail = frame.loc[frame["reverse_minute"] <= tail_minutes].copy()\n    daily = (\n        tail.groupby(["date", "instrument"], sort=False)\n        .agg(\n            spread_close=("relative_spread", "median"),\n            depth_completeness=("depth_completeness", "median"),\n            bid_imbalance=("bid_imbalance", "median"),\n            valid_tail_minutes=("mid_price", "count"),\n        )\n        .reset_index()\n        .merge(recovery, on=["date", "instrument"], how="left")\n    )\n    invalid_tail = daily["valid_tail_minutes"] < min_valid_tail_minutes\n    daily.loc[invalid_tail, list(COMPONENT_COLUMNS)] = np.nan\n    daily[list(COMPONENT_COLUMNS)] = daily[list(COMPONENT_COLUMNS)].replace(\n        [np.inf, -np.inf],\n        np.nan,\n    )\n    return daily.sort_values(["instrument", "date"]).reset_index(drop=True)\n\n\ndef build_ob_001_factor(\n    minute_bars: pd.DataFrame,\n    pool: pd.DataFrame,\n    *,\n    start_date: object | None = None,\n    end_date: object | None = None,\n) -> pd.DataFrame:\n    """Return the exact ``date, instrument, factor`` research interface."""\n\n    _require_columns(pool, POOL_COLUMNS, "pool")\n    daily = compute_ob_001_daily(minute_bars)\n    panel = pool.loc[:, POOL_COLUMNS].copy()\n    panel["date"] = pd.to_datetime(panel["date"], errors="coerce").dt.normalize().astype("datetime64[ns]")\n    panel["instrument"] = panel["instrument"].astype(str)\n    panel = panel.dropna(subset=["date", "instrument"])\n    if start_date is not None:\n        panel = panel.loc[panel["date"] >= pd.Timestamp(start_date).normalize()]\n    if end_date is not None:\n        panel = panel.loc[panel["date"] <= pd.Timestamp(end_date).normalize()]\n    result = panel.merge(\n        daily[["date", "instrument", *COMPONENT_COLUMNS]],\n        on=["date", "instrument"],\n        how="left",\n        validate="one_to_one",\n    )\n\n    oriented: list[pd.Series] = []\n    for column in COMPONENT_COLUMNS:\n        values = pd.to_numeric(result[column], errors="coerce")\n        daily_median = values.groupby(result["date"], sort=False).transform("median")\n        values = values.fillna(daily_median)\n        rank = values.groupby(result["date"], sort=False).rank(pct=True, method="average")\n        if column == "spread_close":\n            rank = 1.0 - rank\n        oriented.append(rank.fillna(0.5))\n    result["factor_raw"] = pd.concat(oriented, axis=1).mean(axis=1)\n    result["factor"] = (\n        result.groupby("date", sort=False)["factor_raw"]\n        .rank(pct=True, method="average")\n        .sub(0.5)\n        .mul(2.0)\n    )\n    if not np.isfinite(result["factor"]).all():\n        raise ValueError("OB-001 produced non-finite factor values")\n    return (\n        result.loc[:, OUTPUT_COLUMNS]\n        .drop_duplicates(["date", "instrument"], keep="last")\n        .sort_values(["date", "instrument"])\n        .reset_index(drop=True)\n    )\n\n\ndef build_ob_001_factor_from_daily(\n    daily_features: pd.DataFrame,\n    pool: pd.DataFrame,\n) -> pd.DataFrame:\n    """Build OB-001 from the frozen AIStudio daily component panel."""\n\n    source_columns = {\n        "spread_close": "tail_60_relative_spread_median",\n        "depth_completeness": "tail_60_depth_completeness_median",\n        "bid_imbalance": "tail_60_bid_depth_imbalance_median",\n        "bid_recovery": "negative_mid_shock_q10_bid_depth_recovery_5m_median",\n    }\n    _require_columns(\n        daily_features,\n        ("date", "instrument", *source_columns.values()),\n        "daily_features",\n    )\n    _require_columns(pool, POOL_COLUMNS, "pool")\n    daily = daily_features[\n        ["date", "instrument", *source_columns.values()]\n    ].rename(columns={value: key for key, value in source_columns.items()})\n    daily["date"] = pd.to_datetime(daily["date"], errors="coerce").dt.normalize()\n    daily["instrument"] = daily["instrument"].astype(str)\n    panel = pool.loc[:, POOL_COLUMNS].copy()\n    panel["date"] = pd.to_datetime(panel["date"], errors="coerce").dt.normalize().astype("datetime64[ns]")\n    panel["instrument"] = panel["instrument"].astype(str)\n    result = panel.merge(\n        daily,\n        on=["date", "instrument"],\n        how="left",\n        validate="one_to_one",\n    )\n    oriented: list[pd.Series] = []\n    for column in COMPONENT_COLUMNS:\n        values = pd.to_numeric(result[column], errors="coerce")\n        values = values.fillna(values.groupby(result["date"], sort=False).transform("median"))\n        rank = values.groupby(result["date"], sort=False).rank(\n            pct=True, method="average"\n        )\n        oriented.append((1.0 - rank if column == "spread_close" else rank).fillna(0.5))\n    result["factor_raw"] = pd.concat(oriented, axis=1).mean(axis=1)\n    result["factor"] = (\n        result.groupby("date", sort=False)["factor_raw"]\n        .rank(pct=True, method="average")\n        .sub(0.5)\n        .mul(2.0)\n    )\n    if not np.isfinite(result["factor"]).all():\n        raise ValueError("OB-001 produced non-finite factor values")\n    return result.loc[:, OUTPUT_COLUMNS].sort_values(\n        ["date", "instrument"]\n    ).reset_index(drop=True)\n', 'bigalpha2026.candidates.ob.ob_002': '"""OB-002: persistent skew in valid order-book depth shape."""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Iterable\n\nimport numpy as np\nimport pandas as pd\n\nLEVELS = range(1, 6)\nPRICE_COLUMNS = tuple(\n    column\n    for level in LEVELS\n    for column in (f"bid_price{level}", f"ask_price{level}")\n)\nVOLUME_COLUMNS = tuple(\n    column\n    for level in LEVELS\n    for column in (f"bid_volume{level}", f"ask_volume{level}")\n)\nBAR_COLUMNS = ("date", "instrument", *PRICE_COLUMNS, *VOLUME_COLUMNS)\nPOOL_COLUMNS = ("date", "instrument")\nOUTPUT_COLUMNS = ("date", "instrument", "factor")\nCOMPONENT_COLUMNS = ("persistent_shape", "full_day_shape")\n\n\ndef _require_columns(frame: pd.DataFrame, columns: Iterable[str], name: str) -> None:\n    missing = sorted(set(columns).difference(frame.columns))\n    if missing:\n        raise ValueError(f"{name} is missing required columns: {missing}")\n\n\ndef compute_ob_002_daily(\n    minute_bars: pd.DataFrame,\n    *,\n    tail_minutes: int = 60,\n    min_valid_tail_minutes: int = 30,\n) -> pd.DataFrame:\n    """Compare near-quote depth concentration on the bid and ask sides."""\n\n    _require_columns(minute_bars, BAR_COLUMNS, "minute_bars")\n    frame = minute_bars.loc[:, BAR_COLUMNS].copy()\n    frame["timestamp"] = pd.to_datetime(frame["date"], errors="coerce")\n    frame["date"] = frame["timestamp"].dt.normalize()\n    frame["instrument"] = frame["instrument"].astype(str)\n    for column in (*PRICE_COLUMNS, *VOLUME_COLUMNS):\n        frame[column] = pd.to_numeric(frame[column], errors="coerce")\n    frame = frame.dropna(subset=["timestamp", "instrument"]).sort_values(\n        ["instrument", "timestamp"]\n    )\n\n    bid_total = pd.Series(0.0, index=frame.index)\n    ask_total = pd.Series(0.0, index=frame.index)\n    bid_near = pd.Series(0.0, index=frame.index)\n    ask_near = pd.Series(0.0, index=frame.index)\n    valid_bid_count = pd.Series(0, index=frame.index)\n    valid_ask_count = pd.Series(0, index=frame.index)\n    for level in LEVELS:\n        valid_bid = (frame[f"bid_price{level}"] > 0) & (frame[f"bid_volume{level}"] > 0)\n        valid_ask = (frame[f"ask_price{level}"] > 0) & (frame[f"ask_volume{level}"] > 0)\n        bid_volume = frame[f"bid_volume{level}"].where(valid_bid, 0.0)\n        ask_volume = frame[f"ask_volume{level}"].where(valid_ask, 0.0)\n        bid_total = bid_total + bid_volume\n        ask_total = ask_total + ask_volume\n        if level <= 2:\n            bid_near = bid_near + bid_volume\n            ask_near = ask_near + ask_volume\n        valid_bid_count = valid_bid_count + valid_bid.astype(int)\n        valid_ask_count = valid_ask_count + valid_ask.astype(int)\n\n    valid_sides = (bid_total > 0) & (ask_total > 0)\n    bid_near_share = (bid_near / bid_total.replace(0, np.nan)).where(valid_sides)\n    ask_near_share = (ask_near / ask_total.replace(0, np.nan)).where(valid_sides)\n    frame["shape_skew"] = bid_near_share - ask_near_share\n    frame["depth_completeness"] = (\n        np.minimum(valid_bid_count, valid_ask_count) / 5.0\n    ).where(valid_sides)\n    day_group = frame.groupby(["date", "instrument"], sort=False)\n    frame["reverse_minute"] = day_group.cumcount(ascending=False) + 1\n\n    full_day = (\n        frame.groupby(["date", "instrument"], sort=False)["shape_skew"]\n        .median()\n        .rename("full_day_shape")\n        .reset_index()\n    )\n    tail = frame.loc[frame["reverse_minute"] <= tail_minutes].copy()\n    tail["shape_sign"] = np.sign(tail["shape_skew"])\n    tail_daily = (\n        tail.groupby(["date", "instrument"], sort=False)\n        .agg(\n            tail_shape=("shape_skew", "median"),\n            sign_mean=("shape_sign", "mean"),\n            valid_tail_minutes=("shape_skew", "count"),\n            depth_completeness=("depth_completeness", "median"),\n        )\n        .reset_index()\n    )\n    tail_daily["persistent_shape"] = tail_daily["tail_shape"] * tail_daily[\n        "sign_mean"\n    ].abs()\n    daily = tail_daily.merge(full_day, on=["date", "instrument"], how="left")\n    invalid = daily["valid_tail_minutes"] < min_valid_tail_minutes\n    daily.loc[invalid, list(COMPONENT_COLUMNS)] = np.nan\n    daily[list(COMPONENT_COLUMNS)] = daily[list(COMPONENT_COLUMNS)].replace(\n        [np.inf, -np.inf],\n        np.nan,\n    )\n    return daily.sort_values(["instrument", "date"]).reset_index(drop=True)\n\n\ndef build_ob_002_factor(\n    minute_bars: pd.DataFrame,\n    pool: pd.DataFrame,\n    *,\n    start_date: object | None = None,\n    end_date: object | None = None,\n) -> pd.DataFrame:\n    """Return the exact ``date, instrument, factor`` research interface."""\n\n    _require_columns(pool, POOL_COLUMNS, "pool")\n    daily = compute_ob_002_daily(minute_bars)\n    panel = pool.loc[:, POOL_COLUMNS].copy()\n    panel["date"] = pd.to_datetime(panel["date"], errors="coerce").dt.normalize().astype("datetime64[ns]")\n    panel["instrument"] = panel["instrument"].astype(str)\n    if start_date is not None:\n        panel = panel.loc[panel["date"] >= pd.Timestamp(start_date).normalize()]\n    if end_date is not None:\n        panel = panel.loc[panel["date"] <= pd.Timestamp(end_date).normalize()]\n    result = panel.merge(\n        daily[["date", "instrument", *COMPONENT_COLUMNS]],\n        on=["date", "instrument"],\n        how="left",\n        validate="one_to_one",\n    )\n    ranks: list[pd.Series] = []\n    for column in COMPONENT_COLUMNS:\n        values = pd.to_numeric(result[column], errors="coerce")\n        median = values.groupby(result["date"], sort=False).transform("median")\n        values = values.fillna(median)\n        ranks.append(\n            values.groupby(result["date"], sort=False)\n            .rank(pct=True, method="average")\n            .fillna(0.5)\n        )\n    result["factor_raw"] = pd.concat(ranks, axis=1).mean(axis=1)\n    result["factor"] = (\n        result.groupby("date", sort=False)["factor_raw"]\n        .rank(pct=True, method="average")\n        .sub(0.5)\n        .mul(2.0)\n    )\n    if not np.isfinite(result["factor"]).all():\n        raise ValueError("OB-002 produced non-finite factor values")\n    return (\n        result.loc[:, OUTPUT_COLUMNS]\n        .drop_duplicates(["date", "instrument"], keep="last")\n        .sort_values(["date", "instrument"])\n        .reset_index(drop=True)\n    )\n\n\ndef build_ob_002_factor_from_daily(\n    daily_features: pd.DataFrame,\n    pool: pd.DataFrame,\n) -> pd.DataFrame:\n    """Build OB-002 from the frozen AIStudio daily component panel."""\n\n    required = (\n        "date",\n        "instrument",\n        "full_day_depth_shape_median",\n        "tail_60_depth_shape_median",\n        "tail_60_shape_sign_consistency",\n    )\n    _require_columns(daily_features, required, "daily_features")\n    _require_columns(pool, POOL_COLUMNS, "pool")\n    daily = daily_features.loc[:, required].copy()\n    daily["date"] = pd.to_datetime(daily["date"], errors="coerce").dt.normalize()\n    daily["instrument"] = daily["instrument"].astype(str)\n    daily["persistent_shape"] = pd.to_numeric(\n        daily["tail_60_depth_shape_median"], errors="coerce"\n    ) * pd.to_numeric(\n        daily["tail_60_shape_sign_consistency"], errors="coerce"\n    ).abs()\n    daily["full_day_shape"] = pd.to_numeric(\n        daily["full_day_depth_shape_median"], errors="coerce"\n    )\n    panel = pool.loc[:, POOL_COLUMNS].copy()\n    panel["date"] = pd.to_datetime(panel["date"], errors="coerce").dt.normalize().astype("datetime64[ns]")\n    panel["instrument"] = panel["instrument"].astype(str)\n    result = panel.merge(\n        daily[["date", "instrument", *COMPONENT_COLUMNS]],\n        on=["date", "instrument"],\n        how="left",\n        validate="one_to_one",\n    )\n    ranks: list[pd.Series] = []\n    for column in COMPONENT_COLUMNS:\n        values = pd.to_numeric(result[column], errors="coerce")\n        values = values.fillna(values.groupby(result["date"], sort=False).transform("median"))\n        ranks.append(\n            values.groupby(result["date"], sort=False)\n            .rank(pct=True, method="average")\n            .fillna(0.5)\n        )\n    result["factor_raw"] = pd.concat(ranks, axis=1).mean(axis=1)\n    result["factor"] = (\n        result.groupby("date", sort=False)["factor_raw"]\n        .rank(pct=True, method="average")\n        .sub(0.5)\n        .mul(2.0)\n    )\n    if not np.isfinite(result["factor"]).all():\n        raise ValueError("OB-002 produced non-finite factor values")\n    return result.loc[:, OUTPUT_COLUMNS].sort_values(\n        ["date", "instrument"]\n    ).reset_index(drop=True)\n', 'bigalpha2026.candidates.ob.ob_003': '"""OB-003: directional order-book resilience asymmetry."""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Iterable\n\nimport numpy as np\nimport pandas as pd\n\nPOOL_COLUMNS = ("date", "instrument")\nOUTPUT_COLUMNS = ("date", "instrument", "factor")\nNEGATIVE_RECOVERY = "negative_mid_shock_q10_bid_depth_recovery_5m_median"\nPOSITIVE_RECOVERY = "positive_mid_shock_q90_ask_depth_recovery_5m_median"\n\n\ndef _require_columns(frame: pd.DataFrame, columns: Iterable[str], name: str) -> None:\n    missing = sorted(set(columns).difference(frame.columns))\n    if missing:\n        raise ValueError(f"{name} is missing required columns: {missing}")\n\n\ndef build_ob_003_factor_from_daily(\n    daily_features: pd.DataFrame,\n    pool: pd.DataFrame,\n) -> pd.DataFrame:\n    """Rank buy-side replenishment against symmetric sell-side replenishment."""\n\n    required = (*POOL_COLUMNS, NEGATIVE_RECOVERY, POSITIVE_RECOVERY)\n    _require_columns(daily_features, required, "daily_features")\n    _require_columns(pool, POOL_COLUMNS, "pool")\n\n    daily = daily_features.loc[:, required].copy()\n    daily["date"] = pd.to_datetime(daily["date"], errors="coerce").dt.normalize()\n    daily["instrument"] = daily["instrument"].astype(str)\n    if daily.duplicated(list(POOL_COLUMNS)).any():\n        raise ValueError("daily_features contains duplicate date-instrument keys")\n\n    panel = pool.loc[:, POOL_COLUMNS].copy()\n    panel["date"] = pd.to_datetime(panel["date"], errors="coerce").dt.normalize().astype("datetime64[ns]")\n    panel["instrument"] = panel["instrument"].astype(str)\n    panel = panel.dropna(subset=list(POOL_COLUMNS))\n    if panel.duplicated(list(POOL_COLUMNS)).any():\n        raise ValueError("pool contains duplicate date-instrument keys")\n\n    result = panel.merge(\n        daily,\n        on=list(POOL_COLUMNS),\n        how="left",\n        validate="one_to_one",\n    )\n    component_ranks: dict[str, pd.Series] = {}\n    for column in (NEGATIVE_RECOVERY, POSITIVE_RECOVERY):\n        values = pd.to_numeric(result[column], errors="coerce").replace(\n            [np.inf, -np.inf],\n            np.nan,\n        )\n        median = values.groupby(result["date"], sort=False).transform("median")\n        values = values.fillna(median)\n        component_ranks[column] = (\n            values.groupby(result["date"], sort=False).rank(pct=True, method="average").fillna(0.5)\n        )\n\n    result["factor_raw"] = component_ranks[NEGATIVE_RECOVERY] - component_ranks[POSITIVE_RECOVERY]\n    result["factor"] = (\n        result.groupby("date", sort=False)["factor_raw"]\n        .rank(pct=True, method="average")\n        .sub(0.5)\n        .mul(2.0)\n    )\n    if not np.isfinite(result["factor"]).all():\n        raise ValueError("OB-003 produced non-finite factor values")\n    return result.loc[:, OUTPUT_COLUMNS].sort_values(["date", "instrument"]).reset_index(drop=True)\n', 'bigalpha2026.candidates.ob.ob_004': '"""OB-004: closing order-book imbalance innovation."""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Iterable\n\nimport numpy as np\nimport pandas as pd\n\nPOOL_COLUMNS = ("date", "instrument")\nOUTPUT_COLUMNS = ("date", "instrument", "factor")\nFULL_DAY_IMBALANCE = "full_day_depth_imbalance_median"\nFULL_DAY_IMBALANCE_STD = "full_day_depth_imbalance_std"\nTAIL_IMBALANCE = "tail_60_bid_depth_imbalance_median"\n\n\ndef _require_columns(frame: pd.DataFrame, columns: Iterable[str], name: str) -> None:\n    missing = sorted(set(columns).difference(frame.columns))\n    if missing:\n        raise ValueError(f"{name} is missing required columns: {missing}")\n\n\ndef compute_ob_004_daily(daily_features: pd.DataFrame) -> pd.DataFrame:\n    """Standardize the closing book-state imbalance against its daily state."""\n\n    required = (\n        *POOL_COLUMNS,\n        FULL_DAY_IMBALANCE,\n        FULL_DAY_IMBALANCE_STD,\n        TAIL_IMBALANCE,\n    )\n    _require_columns(daily_features, required, "daily_features")\n    daily = daily_features.loc[:, required].copy()\n    daily["date"] = pd.to_datetime(daily["date"], errors="coerce").dt.normalize()\n    daily["instrument"] = daily["instrument"].astype(str)\n    if daily.duplicated(list(POOL_COLUMNS)).any():\n        raise ValueError("daily_features contains duplicate date-instrument keys")\n\n    full_day = pd.to_numeric(daily[FULL_DAY_IMBALANCE], errors="coerce")\n    tail = pd.to_numeric(daily[TAIL_IMBALANCE], errors="coerce")\n    dispersion = pd.to_numeric(\n        daily[FULL_DAY_IMBALANCE_STD], errors="coerce"\n    )\n    daily["factor_raw"] = (tail - full_day) / dispersion.where(\n        dispersion > 1e-6\n    )\n    daily["factor_raw"] = daily["factor_raw"].replace(\n        [np.inf, -np.inf],\n        np.nan,\n    )\n    return daily[\n        [\n            "date",\n            "instrument",\n            FULL_DAY_IMBALANCE,\n            TAIL_IMBALANCE,\n            "factor_raw",\n        ]\n    ].sort_values(["instrument", "date"]).reset_index(drop=True)\n\n\ndef build_ob_004_factor_from_daily(\n    daily_features: pd.DataFrame,\n    pool: pd.DataFrame,\n) -> pd.DataFrame:\n    """Build OB-004 from the frozen AIStudio daily microstructure panel."""\n\n    _require_columns(pool, POOL_COLUMNS, "pool")\n    daily = compute_ob_004_daily(daily_features)\n    panel = pool.loc[:, POOL_COLUMNS].copy()\n    panel["date"] = pd.to_datetime(panel["date"], errors="coerce").dt.normalize().astype("datetime64[ns]")\n    panel["instrument"] = panel["instrument"].astype(str)\n    panel = panel.dropna(subset=list(POOL_COLUMNS))\n    if panel.duplicated(list(POOL_COLUMNS)).any():\n        raise ValueError("pool contains duplicate date-instrument keys")\n\n    result = panel.merge(\n        daily[["date", "instrument", "factor_raw"]],\n        on=list(POOL_COLUMNS),\n        how="left",\n        validate="one_to_one",\n    )\n    median = result.groupby("date", sort=False)["factor_raw"].transform("median")\n    result["factor_raw"] = result["factor_raw"].fillna(median).fillna(0.0)\n    result["factor"] = (\n        result.groupby("date", sort=False)["factor_raw"]\n        .rank(pct=True, method="average")\n        .sub(0.5)\n        .mul(2.0)\n    )\n    if not np.isfinite(result["factor"]).all():\n        raise ValueError("OB-004 produced non-finite factor values")\n    return result.loc[:, OUTPUT_COLUMNS].sort_values(\n        ["date", "instrument"]\n    ).reset_index(drop=True)\n', 'bigalpha2026.candidates.ob.ob_005': '"""OB-005: persistent closing microprice pressure."""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Iterable\n\nimport numpy as np\nimport pandas as pd\n\nPOOL_COLUMNS = ("date", "instrument")\nOUTPUT_COLUMNS = ("date", "instrument", "factor")\nTAIL_MICROPRICE_GAP = "tail_60_microprice_gap_median"\nTAIL_MICROPRICE_CONSISTENCY = "tail_60_microprice_gap_sign_consistency"\n\n\ndef _require_columns(frame: pd.DataFrame, columns: Iterable[str], name: str) -> None:\n    missing = sorted(set(columns).difference(frame.columns))\n    if missing:\n        raise ValueError(f"{name} is missing required columns: {missing}")\n\n\ndef compute_ob_005_daily(daily_features: pd.DataFrame) -> pd.DataFrame:\n    """Combine the closing microprice gap with its directional persistence."""\n\n    required = (\n        *POOL_COLUMNS,\n        TAIL_MICROPRICE_GAP,\n        TAIL_MICROPRICE_CONSISTENCY,\n    )\n    _require_columns(daily_features, required, "daily_features")\n    daily = daily_features.loc[:, required].copy()\n    daily["date"] = pd.to_datetime(daily["date"], errors="coerce").dt.normalize()\n    daily["instrument"] = daily["instrument"].astype(str)\n    if daily.duplicated(list(POOL_COLUMNS)).any():\n        raise ValueError("daily_features contains duplicate date-instrument keys")\n\n    gap = pd.to_numeric(daily[TAIL_MICROPRICE_GAP], errors="coerce")\n    consistency = pd.to_numeric(\n        daily[TAIL_MICROPRICE_CONSISTENCY],\n        errors="coerce",\n    ).abs().clip(0.0, 1.0)\n    daily["factor_raw"] = gap.clip(-0.5, 0.5) * consistency\n    daily["factor_raw"] = daily["factor_raw"].replace(\n        [np.inf, -np.inf],\n        np.nan,\n    )\n    return daily[\n        [\n            "date",\n            "instrument",\n            TAIL_MICROPRICE_GAP,\n            TAIL_MICROPRICE_CONSISTENCY,\n            "factor_raw",\n        ]\n    ].sort_values(["instrument", "date"]).reset_index(drop=True)\n\n\ndef build_ob_005_factor_from_daily(\n    daily_features: pd.DataFrame,\n    pool: pd.DataFrame,\n) -> pd.DataFrame:\n    """Build OB-005 from the extended AIStudio daily microstructure panel."""\n\n    _require_columns(pool, POOL_COLUMNS, "pool")\n    daily = compute_ob_005_daily(daily_features)\n    panel = pool.loc[:, POOL_COLUMNS].copy()\n    panel["date"] = pd.to_datetime(panel["date"], errors="coerce").dt.normalize().astype("datetime64[ns]")\n    panel["instrument"] = panel["instrument"].astype(str)\n    panel = panel.dropna(subset=list(POOL_COLUMNS))\n    if panel.duplicated(list(POOL_COLUMNS)).any():\n        raise ValueError("pool contains duplicate date-instrument keys")\n    result = panel.merge(\n        daily[["date", "instrument", "factor_raw"]],\n        on=list(POOL_COLUMNS),\n        how="left",\n        validate="one_to_one",\n    )\n    median = result.groupby("date", sort=False)["factor_raw"].transform("median")\n    result["factor_raw"] = result["factor_raw"].fillna(median).fillna(0.0)\n    result["factor"] = (\n        result.groupby("date", sort=False)["factor_raw"]\n        .rank(pct=True, method="average")\n        .sub(0.5)\n        .mul(2.0)\n    )\n    if not np.isfinite(result["factor"]).all():\n        raise ValueError("OB-005 produced non-finite factor values")\n    return result.loc[:, OUTPUT_COLUMNS].sort_values(\n        ["date", "instrument"]\n    ).reset_index(drop=True)\n', 'bigalpha2026.candidates.composite.int_002': '"""INT-002: earnings-announcement abnormal overnight-return drift."""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom bigalpha2026.candidates.fr._common import (\n    group_asof,\n    prepare_pool,\n    rank_state,\n    require_columns,\n)\nfrom bigalpha2026.candidates.pv._common import prepare_daily\n\nEVENT_COLUMNS = (\n    "disclosure_date",\n    "effective_date",\n    "instrument",\n    "report_date",\n    "category",\n    "shift",\n)\n\n\ndef compute_int_002_events(\n    financial: pd.DataFrame,\n    daily_bars: pd.DataFrame,\n    pool: pd.DataFrame,\n) -> pd.DataFrame:\n    """Measure each report\'s opening reaction relative to the daily universe."""\n\n    require_columns(financial, EVENT_COLUMNS, "financial")\n    events = financial.loc[:, EVENT_COLUMNS].copy()\n    for column in ("disclosure_date", "effective_date", "report_date"):\n        events[column] = pd.to_datetime(events[column], errors="coerce").dt.normalize()\n    events["instrument"] = events["instrument"].astype(str)\n    events["category"] = events["category"].astype(str).str.lower()\n    events["shift"] = pd.to_numeric(events["shift"], errors="coerce")\n    events = events.loc[events["category"].eq("ttm") & events["shift"].eq(0)]\n    events = (\n        events.dropna(\n            subset=[\n                "disclosure_date",\n                "effective_date",\n                "instrument",\n                "report_date",\n            ]\n        )\n        .sort_values(["instrument", "report_date", "disclosure_date"])\n        .drop_duplicates(["instrument", "report_date"], keep="first")\n        .sort_values(["instrument", "effective_date", "report_date"])\n        .drop_duplicates(["instrument", "effective_date"], keep="last")\n        .reset_index(drop=True)\n    )\n\n    bars = prepare_daily(\n        daily_bars,\n        ("date", "instrument", "open", "pre_close"),\n    )\n    universe = prepare_pool(pool)\n    bars = universe.merge(\n        bars,\n        on=["date", "instrument"],\n        how="left",\n        validate="one_to_one",\n    )\n    bars["overnight_return"] = (\n        bars["open"] / bars["pre_close"].where(bars["pre_close"] > 0) - 1.0\n    )\n    bars["market_overnight_return"] = bars.groupby(\n        "date",\n        sort=False,\n    )["overnight_return"].transform("mean")\n    bars["abnormal_overnight_return"] = (\n        bars["overnight_return"] - bars["market_overnight_return"]\n    )\n    events = events.merge(\n        bars[\n            [\n                "date",\n                "instrument",\n                "abnormal_overnight_return",\n            ]\n        ],\n        left_on=["effective_date", "instrument"],\n        right_on=["date", "instrument"],\n        how="left",\n        validate="one_to_one",\n    ).drop(columns="date")\n    events["factor_raw"] = events["abnormal_overnight_return"].replace(\n        [np.inf, -np.inf],\n        np.nan,\n    )\n    return events[\n        [\n            "instrument",\n            "disclosure_date",\n            "effective_date",\n            "report_date",\n            "factor_raw",\n        ]\n    ]\n\n\ndef build_int_002_factor(\n    financial: pd.DataFrame,\n    daily_bars: pd.DataFrame,\n    pool: pd.DataFrame,\n    *,\n    active_days: int = 20,\n) -> pd.DataFrame:\n    """Forward-fill the event reaction for at most ``active_days`` sessions."""\n\n    if active_days < 1:\n        raise ValueError("active_days must be positive")\n    panel = prepare_pool(pool)\n    state = group_asof(\n        panel,\n        compute_int_002_events(financial, daily_bars, panel),\n        left_on="date",\n        right_on="effective_date",\n        right_columns=["factor_raw"],\n    )\n    state["event_age"] = np.nan\n    active = state["effective_date"].notna()\n    state.loc[active, "event_age"] = (\n        state.loc[active]\n        .groupby(["instrument", "effective_date"], sort=False)\n        .cumcount()\n        .astype(float)\n    )\n    state.loc[state["event_age"].ge(active_days), "factor_raw"] = np.nan\n    return rank_state(state, candidate_id="INT-002")\n', 'bigalpha2026.candidates.composite.int_003': '"""INT-003: earnings-surprise drift conditioned on event-day illiquidity."""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom bigalpha2026.candidates.fr._common import (\n    group_asof,\n    prepare_pool,\n    rank_state,\n    require_columns,\n)\nfrom bigalpha2026.candidates.fr.fr_012 import compute_fr_012_events\n\nMICRO_COLUMNS = (\n    "date",\n    "instrument",\n    "full_day_relative_spread_median",\n    "tail_60_relative_spread_median",\n    "full_day_total_depth_median",\n    "tail_60_total_depth_median",\n)\n\n\ndef compute_int_003_events(\n    financial: pd.DataFrame,\n    micro_daily: pd.DataFrame,\n    pool: pd.DataFrame,\n) -> pd.DataFrame:\n    """Scale earnings surprise when event-day liquidity becomes fragile."""\n\n    events = compute_fr_012_events(financial).rename(\n        columns={"factor_raw": "earnings_surprise"}\n    )\n    require_columns(micro_daily, MICRO_COLUMNS, "micro_daily")\n    micro = micro_daily.loc[:, MICRO_COLUMNS].copy()\n    micro["date"] = pd.to_datetime(micro["date"], errors="coerce").dt.normalize()\n    micro["instrument"] = micro["instrument"].astype(str)\n    if micro.duplicated(["date", "instrument"]).any():\n        raise ValueError("micro_daily contains duplicate date-instrument keys")\n    panel = prepare_pool(pool)\n    micro = panel.merge(\n        micro,\n        on=["date", "instrument"],\n        how="left",\n        validate="one_to_one",\n    )\n    for column in MICRO_COLUMNS[2:]:\n        micro[column] = pd.to_numeric(micro[column], errors="coerce")\n    full_spread = micro["full_day_relative_spread_median"].where(\n        micro["full_day_relative_spread_median"] > 0\n    )\n    tail_spread = micro["tail_60_relative_spread_median"].where(\n        micro["tail_60_relative_spread_median"] > 0\n    )\n    full_depth = micro["full_day_total_depth_median"].where(\n        micro["full_day_total_depth_median"] > 0\n    )\n    tail_depth = micro["tail_60_total_depth_median"].where(\n        micro["tail_60_total_depth_median"] > 0\n    )\n    micro["liquidity_shock"] = np.log(tail_spread / full_spread) + np.log(\n        full_depth / tail_depth\n    )\n    liquidity_rank = micro.groupby(\n        "date", sort=False\n    )["liquidity_shock"].rank(pct=True, method="average")\n    micro["positive_liquidity_shock"] = (\n        liquidity_rank.sub(0.5).mul(2.0).clip(lower=0.0).fillna(0.0)\n    )\n\n    events = events.merge(\n        micro[["date", "instrument", "positive_liquidity_shock"]],\n        left_on=["effective_date", "instrument"],\n        right_on=["date", "instrument"],\n        how="left",\n        validate="many_to_one",\n    ).drop(columns="date")\n    events["positive_liquidity_shock"] = events[\n        "positive_liquidity_shock"\n    ].fillna(0.0)\n    events["factor_raw"] = events["earnings_surprise"] * (\n        1.0 + events["positive_liquidity_shock"]\n    )\n    return events[\n        [\n            "instrument",\n            "disclosure_date",\n            "effective_date",\n            "report_date",\n            "factor_raw",\n        ]\n    ]\n\n\ndef build_int_003_factor(\n    financial: pd.DataFrame,\n    micro_daily: pd.DataFrame,\n    pool: pd.DataFrame,\n    *,\n    active_days: int = 20,\n) -> pd.DataFrame:\n    """Forward-fill the event interaction for at most ``active_days`` sessions."""\n\n    if active_days < 1:\n        raise ValueError("active_days must be positive")\n    panel = prepare_pool(pool)\n    state = group_asof(\n        panel,\n        compute_int_003_events(financial, micro_daily, panel),\n        left_on="date",\n        right_on="effective_date",\n        right_columns=["factor_raw"],\n    )\n    state["event_age"] = np.nan\n    active = state["effective_date"].notna()\n    state.loc[active, "event_age"] = (\n        state.loc[active]\n        .groupby(["instrument", "effective_date"], sort=False)\n        .cumcount()\n        .astype(float)\n    )\n    state.loc[state["event_age"].ge(active_days), "factor_raw"] = np.nan\n    return rank_state(state, candidate_id="INT-003")\n'}
    for package in ['bigalpha2026', 'bigalpha2026.candidates', 'bigalpha2026.candidates.fr', 'bigalpha2026.candidates.pv', 'bigalpha2026.candidates.hf', 'bigalpha2026.candidates.ob', 'bigalpha2026.candidates.composite']:
        if package not in sys.modules:
            module = types.ModuleType(package)
            module.__path__ = []
            sys.modules[package] = module
            if '.' in package:
                parent, child = package.rsplit('.', 1)
                setattr(sys.modules[parent], child, module)
    for name, source in sources.items():
        module = types.ModuleType(name)
        module.__package__ = name.rsplit('.', 1)[0]
        sys.modules[name] = module
        parent, child = name.rsplit('.', 1)
        setattr(sys.modules[parent], child, module)
        exec(compile(source, name, 'exec'), module.__dict__)  # noqa: S102

def _rank_center(values, dates, np):
    numeric = values.replace([np.inf, -np.inf], np.nan)
    grouped = numeric.groupby(dates, sort=False)
    ranks = grouped.rank(method='average')
    counts = grouped.transform('count')
    return (2.0 * (ranks - (counts + 1.0) / 2.0) / counts).fillna(0.0)

def _apply_financial_effective_dates(financial, pool, pd, np):
    financial = financial.copy()
    financial['date'] = pd.to_datetime(financial['date'], errors='coerce').dt.normalize()
    financial['disclosure_date'] = financial['date']
    calendar = pd.DatetimeIndex(sorted(pd.to_datetime(pool['date'], errors='coerce').dropna().unique()))
    disclosure_values = financial['disclosure_date'].to_numpy(dtype='datetime64[ns]')
    positions = np.searchsorted(calendar.to_numpy(dtype='datetime64[ns]'), disclosure_values, side='right')
    valid = positions < len(calendar)
    financial['effective_date'] = pd.NaT
    financial.loc[valid, 'effective_date'] = calendar[positions[valid]]
    return financial.drop(columns=['date'])

def _candidate_factors(selected, financial, factorlib, exposure, micro_daily, pool):
    _install_bigalpha_candidate_modules()
    from bigalpha2026.candidates.composite.int_002 import build_int_002_factor
    from bigalpha2026.candidates.composite.int_003 import build_int_003_factor
    from bigalpha2026.candidates.fr.fr_001 import build_fr_001_factor_from_panel
    from bigalpha2026.candidates.fr.fr_002 import build_fr_002_factor_from_panel
    from bigalpha2026.candidates.fr.fr_004 import build_fr_004_factor
    from bigalpha2026.candidates.fr.fr_005 import build_fr_005_factor
    from bigalpha2026.candidates.fr.fr_006 import build_fr_006_factor
    from bigalpha2026.candidates.fr.fr_010 import build_fr_010_factor
    from bigalpha2026.candidates.fr.fr_011 import build_fr_011_factor
    from bigalpha2026.candidates.fr.fr_012 import build_fr_012_factor
    from bigalpha2026.candidates.fr.fr_013 import build_fr_013_factor
    from bigalpha2026.candidates.fr.fr_014 import build_fr_014_factor
    from bigalpha2026.candidates.fr.fr_015 import build_fr_015_factor
    from bigalpha2026.candidates.hf.hf_001 import build_hf_001_factor_from_daily
    from bigalpha2026.candidates.hf.hf_003 import build_hf_003_factor_from_daily
    from bigalpha2026.candidates.hf.hf_004 import build_hf_004_factor_from_daily
    from bigalpha2026.candidates.ob.ob_001 import build_ob_001_factor_from_daily
    from bigalpha2026.candidates.ob.ob_002 import build_ob_002_factor_from_daily
    from bigalpha2026.candidates.ob.ob_003 import build_ob_003_factor_from_daily
    from bigalpha2026.candidates.ob.ob_004 import build_ob_004_factor_from_daily
    from bigalpha2026.candidates.ob.ob_005 import build_ob_005_factor_from_daily
    from bigalpha2026.candidates.pv.pv_002 import build_pv_002_factor
    from bigalpha2026.candidates.pv.pv_003 import build_pv_003_factor
    from bigalpha2026.candidates.pv.pv_004 import build_pv_004_factor
    from bigalpha2026.candidates.pv.pv_005 import build_pv_005_factor
    from bigalpha2026.candidates.pv.pv_008 import build_pv_008_factor
    from bigalpha2026.candidates.pv.pv_009 import build_pv_009_factor
    from bigalpha2026.candidates.pv.pv_010 import build_pv_010_factor
    from bigalpha2026.candidates.pv.pv_011 import build_pv_011_factor
    from bigalpha2026.candidates.pv.pv_013 import build_pv_013_factor
    from bigalpha2026.candidates.pv.pv_014 import build_pv_014_factor
    from bigalpha2026.candidates.pv.pv_015 import build_pv_015_factor
    from bigalpha2026.candidates.pv.pv_016 import build_pv_016_factor
    from bigalpha2026.candidates.pv.pv_017 import build_pv_017_factor
    from bigalpha2026.candidates.pv.pv_019 import build_pv_019_factor
    from bigalpha2026.candidates.pv.pv_020 import build_pv_020_factor
    from bigalpha2026.candidates.pv.pv_021 import build_pv_021_factor
    from bigalpha2026.candidates.pv.pv_022 import build_pv_022_factor
    from bigalpha2026.candidates.pv.pv_023 import build_pv_023_factor
    builders = {
        'FR-001': lambda: build_fr_001_factor_from_panel(financial, pool), 'FR-002': lambda: build_fr_002_factor_from_panel(financial, pool),
        'FR-004': lambda: build_fr_004_factor(financial, pool), 'FR-005': lambda: build_fr_005_factor(financial, exposure, pool),
        'FR-006': lambda: build_fr_006_factor(financial, pool), 'FR-010': lambda: build_fr_010_factor(financial, pool),
        'FR-011': lambda: build_fr_011_factor(financial, exposure, pool), 'FR-012': lambda: build_fr_012_factor(financial, pool),
        'FR-013': lambda: build_fr_013_factor(financial, pool), 'FR-014': lambda: build_fr_014_factor(financial, exposure, pool), 'FR-015': lambda: build_fr_015_factor(financial, pool),
        'PV-002': lambda: build_pv_002_factor(micro_daily, pool), 'PV-003': lambda: build_pv_003_factor(micro_daily, pool),
        'PV-004': lambda: build_pv_004_factor(micro_daily, pool), 'PV-005': lambda: build_pv_005_factor(micro_daily, pool),
        'PV-008': lambda: build_pv_008_factor(micro_daily, pool), 'PV-009': lambda: build_pv_009_factor(micro_daily, pool),
        'PV-010': lambda: build_pv_010_factor(micro_daily, pool), 'PV-011': lambda: build_pv_011_factor(micro_daily, pool),
        'PV-013': lambda: build_pv_013_factor(micro_daily, pool), 'PV-014': lambda: build_pv_014_factor(micro_daily, pool),
        'PV-015': lambda: build_pv_015_factor(micro_daily, pool), 'PV-016': lambda: build_pv_016_factor(factorlib, pool),
        'PV-017': lambda: build_pv_017_factor(micro_daily, pool), 'PV-019': lambda: build_pv_019_factor(micro_daily, pool),
        'PV-020': lambda: build_pv_020_factor(micro_daily, pool), 'PV-021': lambda: build_pv_021_factor(micro_daily, pool),
        'PV-022': lambda: build_pv_022_factor(micro_daily, pool), 'PV-023': lambda: build_pv_023_factor(micro_daily, pool),
        'HF-001': lambda: build_hf_001_factor_from_daily(micro_daily, pool), 'HF-003': lambda: build_hf_003_factor_from_daily(micro_daily, pool), 'HF-004': lambda: build_hf_004_factor_from_daily(micro_daily, pool),
        'OB-001': lambda: build_ob_001_factor_from_daily(micro_daily, pool), 'OB-002': lambda: build_ob_002_factor_from_daily(micro_daily, pool),
        'OB-003': lambda: build_ob_003_factor_from_daily(micro_daily, pool), 'OB-004': lambda: build_ob_004_factor_from_daily(micro_daily, pool), 'OB-005': lambda: build_ob_005_factor_from_daily(micro_daily, pool),
        'INT-002': lambda: build_int_002_factor(financial, micro_daily, pool), 'INT-003': lambda: build_int_003_factor(financial, micro_daily, pool),
    }
    return {cid: builders[cid]() for cid in selected}

def _query_micro_daily(dai, pd, source, history_start, end_ts):
    sql = f"""
        WITH base AS (
            SELECT
                date AS timestamp, instrument, CAST(date_trunc('day', date) AS DATE) AS trading_day,
                CASE WHEN EXTRACT(hour FROM date) < 12 THEN 0 ELSE 1 END AS session_id,
                open, high, low, close, pre_close, amount, volume, deal_number,
                CASE WHEN bid_price1 > 0 AND ask_price1 >= bid_price1 AND bid_volume1 > 0 AND ask_volume1 > 0 THEN (bid_price1 + ask_price1) / 2.0 ELSE NULL END AS mid_price,
                CASE WHEN bid_price1 > 0 AND ask_price1 >= bid_price1 AND bid_volume1 > 0 AND ask_volume1 > 0 THEN (ask_price1 - bid_price1) / ((bid_price1 + ask_price1) / 2.0) ELSE NULL END AS relative_spread,
                CASE WHEN bid_price1 > 0 AND ask_price1 > bid_price1 AND bid_volume1 > 0 AND ask_volume1 > 0 THEN (((ask_price1 * bid_volume1 + bid_price1 * ask_volume1) / (bid_volume1 + ask_volume1)) - (bid_price1 + ask_price1) / 2.0) / (ask_price1 - bid_price1) ELSE NULL END AS microprice_gap,
                (CASE WHEN bid_price1 > 0 AND bid_volume1 > 0 THEN bid_volume1 ELSE 0 END + CASE WHEN bid_price2 > 0 AND bid_volume2 > 0 THEN bid_volume2 ELSE 0 END + CASE WHEN bid_price3 > 0 AND bid_volume3 > 0 THEN bid_volume3 ELSE 0 END + CASE WHEN bid_price4 > 0 AND bid_volume4 > 0 THEN bid_volume4 ELSE 0 END + CASE WHEN bid_price5 > 0 AND bid_volume5 > 0 THEN bid_volume5 ELSE 0 END) AS bid_depth,
                (CASE WHEN ask_price1 > 0 AND ask_volume1 > 0 THEN ask_volume1 ELSE 0 END + CASE WHEN ask_price2 > 0 AND ask_volume2 > 0 THEN ask_volume2 ELSE 0 END + CASE WHEN ask_price3 > 0 AND ask_volume3 > 0 THEN ask_volume3 ELSE 0 END + CASE WHEN ask_price4 > 0 AND ask_volume4 > 0 THEN ask_volume4 ELSE 0 END + CASE WHEN ask_price5 > 0 AND ask_volume5 > 0 THEN ask_volume5 ELSE 0 END) AS ask_depth,
                (CASE WHEN bid_price1 > 0 AND bid_volume1 > 0 THEN bid_volume1 ELSE 0 END + CASE WHEN bid_price2 > 0 AND bid_volume2 > 0 THEN bid_volume2 ELSE 0 END) AS bid_near_depth,
                (CASE WHEN ask_price1 > 0 AND ask_volume1 > 0 THEN ask_volume1 ELSE 0 END + CASE WHEN ask_price2 > 0 AND ask_volume2 > 0 THEN ask_volume2 ELSE 0 END) AS ask_near_depth,
                (CASE WHEN bid_price1 > 0 AND bid_volume1 > 0 THEN 1 ELSE 0 END + CASE WHEN bid_price2 > 0 AND bid_volume2 > 0 THEN 1 ELSE 0 END + CASE WHEN bid_price3 > 0 AND bid_volume3 > 0 THEN 1 ELSE 0 END + CASE WHEN bid_price4 > 0 AND bid_volume4 > 0 THEN 1 ELSE 0 END + CASE WHEN bid_price5 > 0 AND bid_volume5 > 0 THEN 1 ELSE 0 END) AS valid_bid_count,
                (CASE WHEN ask_price1 > 0 AND ask_volume1 > 0 THEN 1 ELSE 0 END + CASE WHEN ask_price2 > 0 AND ask_volume2 > 0 THEN 1 ELSE 0 END + CASE WHEN ask_price3 > 0 AND ask_volume3 > 0 THEN 1 ELSE 0 END + CASE WHEN ask_price4 > 0 AND ask_volume4 > 0 THEN 1 ELSE 0 END + CASE WHEN ask_price5 > 0 AND ask_volume5 > 0 THEN 1 ELSE 0 END) AS valid_ask_count
            FROM {source}
        ), derived AS (
            SELECT *, LEAST(valid_bid_count, valid_ask_count) / 5.0 AS depth_completeness,
                (bid_depth - ask_depth) / NULLIF(bid_depth + ask_depth, 0) AS depth_imbalance,
                bid_near_depth / NULLIF(bid_depth, 0) - ask_near_depth / NULLIF(ask_depth, 0) AS depth_shape,
                (ln(1.0 + GREATEST(COALESCE(amount, 0), 0)) + ln(1.0 + GREATEST(COALESCE(volume, 0), 0)) + ln(1.0 + GREATEST(COALESCE(deal_number, 0), 0))) / 3.0 AS activity
            FROM base
        ), sequenced AS (
            SELECT *,
                CASE WHEN close > 0 AND lag(close) OVER (PARTITION BY instrument, trading_day, session_id ORDER BY timestamp) > 0 THEN close / lag(close) OVER (PARTITION BY instrument, trading_day, session_id ORDER BY timestamp) - 1.0 ELSE NULL END AS minute_return,
                ln(NULLIF(GREATEST(close, 0), 0) / NULLIF(GREATEST(lag(close) OVER (PARTITION BY instrument, trading_day, session_id ORDER BY timestamp), 0), 0)) AS minute_log_return,
                lead(close, 5) OVER (PARTITION BY instrument, trading_day, session_id ORDER BY timestamp) / NULLIF(close, 0) - 1.0 AS future_return_5m,
                lead(close, 15) OVER (PARTITION BY instrument, trading_day, session_id ORDER BY timestamp) / NULLIF(close, 0) - 1.0 AS future_return_15m,
                mid_price / NULLIF(lag(mid_price) OVER (PARTITION BY instrument, trading_day, session_id ORDER BY timestamp), 0) - 1.0 AS mid_return,
                lead(bid_depth, 5) OVER (PARTITION BY instrument, trading_day, session_id ORDER BY timestamp) / NULLIF(bid_depth, 0) - 1.0 AS bid_recovery_5m,
                lead(ask_depth, 5) OVER (PARTITION BY instrument, trading_day, session_id ORDER BY timestamp) / NULLIF(ask_depth, 0) - 1.0 AS ask_recovery_5m,
                row_number() OVER (PARTITION BY instrument, trading_day ORDER BY timestamp DESC) AS reverse_minute
            FROM derived
        ), thresholds AS (
            SELECT *, quantile_cont(abs(minute_return), 0.90) OVER (PARTITION BY instrument, trading_day) AS shock_q90,
                median(activity) OVER (PARTITION BY instrument, trading_day) AS activity_median,
                quantile_cont(mid_return, 0.10) OVER (PARTITION BY instrument, trading_day) AS negative_mid_q10,
                quantile_cont(mid_return, 0.90) OVER (PARTITION BY instrument, trading_day) AS positive_mid_q90,
                stddev_samp(minute_log_return) OVER (PARTITION BY instrument, trading_day) AS intraday_return_sigma
            FROM sequenced
        )
        SELECT CAST(trading_day AS DATETIME) AS date, instrument,
            first(open ORDER BY timestamp) AS open, max(high) AS high, min(low) AS low, last(close ORDER BY timestamp) AS close, first(pre_close ORDER BY timestamp) AS pre_close,
            sum(amount) AS amount, sum(volume) AS volume, sum(deal_number) AS deal_number,
            count(*) AS minute_count, sum(amount) AS total_amount, sum(volume) AS total_volume, sum(deal_number) AS total_deal_number,
            sum(minute_log_return) AS net_log_return, sum(abs(minute_log_return)) AS absolute_log_return,
            sqrt(sum(minute_log_return * minute_log_return)) AS realized_volatility,
            sqrt(sum(CASE WHEN minute_log_return < 0 THEN minute_log_return * minute_log_return ELSE 0 END)) AS downside_realized_volatility,
            max(minute_return) AS max_minute_return, min(minute_return) AS min_minute_return, max(close) / NULLIF(min(close), 0) - 1.0 AS intraday_close_range,
            sum(CASE WHEN reverse_minute <= 30 THEN amount ELSE 0 END) AS tail_30_amount,
            sum(CASE WHEN reverse_minute <= 60 THEN amount ELSE 0 END) AS tail_60_amount,
            sum(CASE WHEN reverse_minute <= 120 THEN amount ELSE 0 END) AS tail_120_amount,
            sum(CASE WHEN reverse_minute <= 60 THEN volume ELSE 0 END) AS tail_60_volume,
            sum(CASE WHEN reverse_minute <= 60 THEN deal_number ELSE 0 END) AS tail_60_deal_number,
            sum(CASE WHEN reverse_minute <= 60 THEN minute_log_return END) AS tail_60_log_return,
            sum(CASE WHEN reverse_minute <= 60 AND volume >= 0 AND minute_log_return IS NOT NULL AND intraday_return_sigma > 0 THEN volume * (2.0 / (1.0 + exp(-1.702 * GREATEST(-6.0, LEAST(6.0, minute_log_return / intraday_return_sigma)))) - 1.0) END) AS tail_60_signed_volume_bvc,
            sum(CASE WHEN session_id = 0 THEN amount ELSE 0 END) / NULLIF(sum(amount), 0) AS morning_amount_share,
            sum(CASE WHEN session_id = 1 THEN abs(minute_log_return) ELSE 0 END) / NULLIF(sum(abs(minute_log_return)), 0) AS afternoon_absolute_return_share,
            sum(amount) / NULLIF(sum(deal_number), 0) AS avg_trade_value, sum(volume) / NULLIF(sum(deal_number), 0) AS avg_trade_volume,
            abs(sum(minute_log_return)) / NULLIF(sum(abs(minute_log_return)), 0) AS directional_efficiency,
            (sum(CASE WHEN reverse_minute <= 60 THEN amount ELSE 0 END) / NULLIF(sum(CASE WHEN reverse_minute <= 60 THEN deal_number ELSE 0 END), 0)) / NULLIF(sum(amount) / NULLIF(sum(deal_number), 0), 0) AS tail_trade_value_ratio,
            sum(CASE WHEN abs(minute_return) >= shock_q90 AND activity >= activity_median AND future_return_5m IS NOT NULL THEN 1 ELSE 0 END) AS shock_q90_active_count,
            avg(CASE WHEN abs(minute_return) >= shock_q90 AND activity >= activity_median THEN abs(minute_return) END) AS shock_q90_mean_abs_return,
            median(CASE WHEN abs(minute_return) >= shock_q90 AND activity >= activity_median AND minute_return <> 0 THEN GREATEST(-2.0, LEAST(2.0, -sign(minute_return) * future_return_5m / abs(minute_return))) END) AS shock_q90_recovery_5m_median,
            median(CASE WHEN abs(minute_return) >= shock_q90 AND activity >= activity_median AND minute_return <> 0 THEN GREATEST(-2.0, LEAST(2.0, -sign(minute_return) * future_return_15m / abs(minute_return))) END) AS shock_q90_recovery_15m_median,
            sum(CASE WHEN mid_price IS NOT NULL THEN 1 ELSE 0 END) AS valid_snapshot_count,
            avg(CASE WHEN mid_price IS NOT NULL THEN 1.0 ELSE 0.0 END) AS both_sides_valid_rate,
            avg(CASE WHEN valid_bid_count = 5 AND valid_ask_count = 5 THEN 1.0 ELSE 0.0 END) AS full_five_levels_rate,
            sum(CASE WHEN reverse_minute <= 60 AND mid_price IS NOT NULL THEN 1 ELSE 0 END) AS tail_60_valid_best_quote_minutes,
            median(relative_spread) AS full_day_relative_spread_median,
            quantile_cont(relative_spread, 0.90) AS full_day_relative_spread_q90,
            median(CASE WHEN reverse_minute <= 60 THEN relative_spread END) AS tail_60_relative_spread_median,
            median(depth_completeness) AS full_day_depth_completeness_median,
            median(CASE WHEN reverse_minute <= 60 THEN depth_completeness END) AS tail_60_depth_completeness_median,
            median(bid_depth + ask_depth) AS full_day_total_depth_median,
            median(CASE WHEN reverse_minute <= 60 THEN bid_depth + ask_depth END) AS tail_60_total_depth_median,
            median(depth_imbalance) AS full_day_depth_imbalance_median,
            stddev_samp(depth_imbalance) AS full_day_depth_imbalance_std,
            median(CASE WHEN reverse_minute <= 60 THEN depth_imbalance END) AS tail_60_bid_depth_imbalance_median,
            median(CASE WHEN reverse_minute <= 60 THEN microprice_gap END) AS tail_60_microprice_gap_median,
            avg(CASE WHEN reverse_minute <= 60 THEN sign(microprice_gap) END) AS tail_60_microprice_gap_sign_consistency,
            median(CASE WHEN mid_return < 0 AND mid_return <= negative_mid_q10 THEN GREATEST(-2.0, LEAST(2.0, bid_recovery_5m)) END) AS negative_mid_shock_q10_bid_depth_recovery_5m_median,
            median(CASE WHEN mid_return > 0 AND mid_return >= positive_mid_q90 THEN GREATEST(-2.0, LEAST(2.0, ask_recovery_5m)) END) AS positive_mid_shock_q90_ask_depth_recovery_5m_median,
            median(depth_shape) AS full_day_depth_shape_median,
            median(CASE WHEN reverse_minute <= 60 THEN depth_shape END) AS tail_60_depth_shape_median,
            avg(CASE WHEN reverse_minute <= 60 THEN sign(depth_shape) END) AS tail_60_shape_sign_consistency
        FROM thresholds GROUP BY trading_day, instrument ORDER BY date, instrument
    """
    parts = []
    cursor = history_start.to_period('M')
    final_period = end_ts.to_period('M')
    while cursor <= final_period:
        month_start = max(history_start, cursor.start_time.normalize())
        month_end = end_ts if cursor == final_period else cursor.end_time.normalize()
        part = dai.query(sql, filters={'date': [month_start.strftime('%Y-%m-%d 00:00:00'), month_end.strftime('%Y-%m-%d 23:59:59')]}, compression=True).df()
        if not part.empty:
            parts.append(part)
        cursor += 1
    if not parts:
        raise ValueError('micro daily query returned no rows')
    return pd.concat(parts, ignore_index=True)


def _load_common_inputs(datasources, start_date, end_date, pd, np):
    start_ts = pd.Timestamp(start_date).normalize(); end_ts = pd.Timestamp(end_date).normalize()
    history_start = start_ts - pd.Timedelta(days=500); financial_start = start_ts - pd.Timedelta(days=2200)
    financial_source = datasources.get('financial', 'bigalpha_2026_financial'); bar1m_source = datasources.get('bar1m', 'bigalpha_2026_stock_bar1m')
    import dai
    public_columns = ('amount', 'atr_14', 'bias_20', 'cci_14', 'float_market_cap', 'kdj_d_9_3_3', 'macd_diff_12_26_9', 'macd_hist_12_26_9', 'momentum_5', 'net_profit_rate_ttm', 'netflow_amount_rate_main', 'total_market_cap', 'turn', 'volatility_5', 'volume')
    pool = dai.query('SELECT date, instrument FROM bigalpha_2026_instruments', filters={'date': [history_start, end_ts]}, compression=True).df()
    factorlib = dai.query(f"SELECT date, instrument, daily_return, {', '.join(public_columns)} FROM bigalpha_2026_factorlib", filters={'date': [history_start, end_ts]}, compression=True).df()
    exposure = dai.query('SELECT date, instrument, float_market_cap FROM bigalpha_2026_exposure', filters={'date': [history_start, end_ts]}, compression=True).df()
    financial = dai.query(f'SELECT date, instrument, category, shift, report_date, net_cffoa, net_profit, operating_revenue, total_assets FROM {financial_source}', filters={'date': [financial_start, end_ts]}, compression=True).df()
    micro_daily = _query_micro_daily(dai, pd, bar1m_source, history_start, end_ts)
    for frame in (pool, factorlib, exposure, micro_daily):
        frame['date'] = pd.to_datetime(frame['date'], errors='coerce').dt.normalize(); frame['instrument'] = frame['instrument'].astype(str)
    pool = pool.dropna(subset=['date', 'instrument']).drop_duplicates(['date', 'instrument'], keep='last').sort_values(['date', 'instrument']).reset_index(drop=True)
    factorlib = factorlib.dropna(subset=['date', 'instrument']).drop_duplicates(['date', 'instrument'], keep='last').sort_values(['date', 'instrument']).reset_index(drop=True)
    exposure = exposure.dropna(subset=['date', 'instrument']).drop_duplicates(['date', 'instrument'], keep='last').sort_values(['date', 'instrument']).reset_index(drop=True)
    micro_daily = micro_daily.dropna(subset=['date', 'instrument']).drop_duplicates(['date', 'instrument'], keep='last').sort_values(['date', 'instrument']).reset_index(drop=True)
    financial = _apply_financial_effective_dates(financial, pool, pd, np)
    return start_ts, end_ts, public_columns, pool, factorlib, exposure, financial, micro_daily

def main(datasources, start_date, end_date):
    import numpy as np
    import pandas as pd
    from sklearn.linear_model import ElasticNet
    self_columns = ['FR-002', 'FR-004', 'FR-005', 'FR-006', 'FR-010', 'FR-011', 'FR-012', 'FR-013', 'FR-014', 'FR-015', 'HF-001', 'HF-003', 'INT-002', 'INT-003', 'OB-001', 'OB-003', 'OB-004', 'PV-003', 'PV-004', 'PV-005', 'PV-008', 'PV-009', 'PV-010', 'PV-011', 'PV-013', 'PV-014', 'PV-015', 'PV-016', 'PV-017', 'PV-019', 'PV-020', 'PV-021', 'PV-022', 'HF-004']
    start_ts, end_ts, public_columns, pool, factorlib, exposure, financial, micro_daily = _load_common_inputs(datasources, start_date, end_date, pd, np)

    public_directions = {'amount': -1.0, 'atr_14': -1.0, 'bias_20': -1.0, 'cci_14': -1.0, 'float_market_cap': -1.0, 'kdj_d_9_3_3': -1.0, 'macd_diff_12_26_9': -1.0, 'macd_hist_12_26_9': -1.0, 'momentum_5': -1.0, 'net_profit_rate_ttm': 1.0, 'netflow_amount_rate_main': -1.0, 'total_market_cap': -1.0, 'turn': -1.0, 'volatility_5': -1.0, 'volume': -1.0}
    factors = _candidate_factors(self_columns, financial, factorlib, exposure, micro_daily, pool)
    long_parts = []
    for cid, frame in factors.items():
        part = frame[['date', 'instrument', 'factor']].copy(); part['candidate_id'] = cid; long_parts.append(part)
    candidate_pool = pd.concat(long_parts, ignore_index=True)
    candidate_wide = candidate_pool.pivot(index=['date', 'instrument'], columns='candidate_id', values='factor').reset_index()
    panel = pool.merge(factorlib[['date', 'instrument', 'daily_return', *public_columns]], on=['date', 'instrument'], how='left', validate='one_to_one').merge(candidate_wide, on=['date', 'instrument'], how='left', validate='one_to_one')
    for column in public_columns:
        panel[column] = _rank_center(pd.to_numeric(panel[column], errors='coerce'), panel['date'], np) * public_directions[column]
    for column in self_columns:
        panel[column] = _rank_center(pd.to_numeric(panel[column], errors='coerce'), panel['date'], np)
    feature_columns = (*public_columns, *self_columns)
    panel = panel.sort_values(['instrument', 'date']).reset_index(drop=True)
    panel['stock_return'] = pd.to_numeric(panel['daily_return'], errors='coerce').replace([np.inf, -np.inf], np.nan)
    all_dates = pd.DatetimeIndex(sorted(panel['date'].dropna().unique()))
    label_calendar = pd.DataFrame({'label_observed_date': all_dates[1:], 'date': all_dates[:-1]})
    target_frame = panel[['date', 'instrument', 'stock_return']].rename(columns={'date': 'label_observed_date', 'stock_return': 'target_raw'}).merge(label_calendar, on='label_observed_date', how='inner', validate='many_to_one')[['date', 'instrument', 'target_raw']]
    panel = panel.merge(target_frame, on=['date', 'instrument'], how='left', validate='one_to_one')
    target_values = pd.to_numeric(panel['target_raw'], errors='coerce')
    panel['target'] = _rank_center(target_values, panel['date'], np).where(target_values.notna())
    prediction_dates = all_dates[(all_dates >= start_ts) & (all_dates <= end_ts)]
    if prediction_dates.empty: raise ValueError('no prediction dates inside the requested window')
    first_test_position = all_dates.get_loc(prediction_dates[0])
    train_end_position = first_test_position - 1
    train_start_position = train_end_position - 60
    if train_start_position < 0: raise ValueError('not enough pre-start history for model training')
    train_dates = all_dates[train_start_position:train_end_position]
    train = panel.loc[panel['date'].isin(train_dates) & panel['target'].notna()]
    test = panel.loc[panel['date'].isin(prediction_dates)]
    if train.empty or test.empty: raise ValueError('empty train or prediction sample')

    model = ElasticNet(alpha=0.001, l1_ratio=0.5, fit_intercept=True, max_iter=20000, random_state=0, positive=True)
    model.fit(train.loc[:, list(feature_columns)].to_numpy(dtype=float), train['target'].to_numpy(dtype=float))
    result = test[['date', 'instrument']].copy()
    result['factor'] = model.predict(test.loc[:, list(feature_columns)].to_numpy(dtype=float))
    result['factor'] = result.groupby('date', sort=False)['factor'].rank(pct=True, method='average').sub(0.5).mul(2.0)
    result['factor'] = pd.to_numeric(result['factor'], errors='coerce').replace([np.inf, -np.inf], np.nan)
    return result.dropna(subset=['date', 'instrument', 'factor']).sort_values(['date', 'instrument']).reset_index(drop=True)
